# INFOSYS SPRINGBOARD INTERNSHIP 7.0
## Batch 4 · Milestone 1 — User Authentication Module
**Tech Stack:** Streamlit + JWT + ngrok + Gmail OTP

Run each cell in order from top to bottom. Make sure to set up your secrets in Google Colab (Secrets 🔑 icon in the sidebar):
- `NGROK_AUTHTOKEN`
- `JWT_SECRET`
- `EMAIL_ADDRESS` (or `EMAIL_ID`)
- `EMAIL_PASSWORD`

In [ ]:
# @title Install Dependencies
# Step 1: Install required dependencies
!pip install -q streamlit pyngrok bcrypt pyjwt python-dotenv pandas


In [ ]:
# @title UI components
# Step 2: Create directories for UI components
!mkdir -p login_component signup_component signup_otp_component signup_complete_component otp_verification_component reset_email_component reset_password_component security_recovery_component


In [ ]:
# @title Login Component
%%writefile login_component/index.html
<!doctype html>
<html>

<head>
  <meta charset="utf-8">
  <style>
    * {
      box-sizing: border-box
    }

    html,
    body {
      margin: 0;
      min-height: 810px;
      font-family: Arial, sans-serif
    }

    body {
      display: grid;
      place-items: center;
      background: transparent
    }

    .card {
      width: min(1000px, 94vw);
      min-height: 710px;
      background: #fff;
      border-radius: 16px;
      padding: 58px;
      text-align: center;
      color: #17233b;
      box-shadow: 0 20px 50px #0003
    }

    .icon {
      margin: 68px auto 22px;
      width: 58px;
      height: 58px;
      border-radius: 50%;
      display: grid;
      place-items: center;
      background: linear-gradient(135deg, #6260e6, #8343ad);
      color: #fff;
      font-size: 24px;
      box-shadow: 0 8px 20px #625ee655
    }

    .card h2 {
      margin: 0 0 10px;
      font-size: 26px
    }

    .card p {
      margin: 0;
      color: #64748b
    }

    .form {
      width: 535px;
      max-width: 100%;
      margin: 36px auto 28px;
      text-align: left
    }

    label {
      display: block;
      font-weight: bold;
      font-size: 13px;
      margin: 17px 0 8px
    }

    .wrap {
      position: relative
    }

    .field {
      width: 100%;
      height: 58px;
      border: 1.5px solid #d6dfec;
      border-radius: 8px;
      padding: 0 48px 0 45px;
      font-size: 15px
    }

    .symbol {
      position: absolute;
      left: 15px;
      top: 18px;
      color: #64748b
    }

    .eye {
      position: absolute;
      right: 12px;
      top: 14px;
      border: 0;
      background: none;
      color: #64748b;
      cursor: pointer;
      font-size: 20px
    }

    .primary {
      width: 100%;
      height: 58px;
      margin-top: 26px;
      border: 0;
      border-radius: 8px;
      background: linear-gradient(90deg, #6260e6, #8343ad);
      color: #fff;
      font-weight: bold;
      font-size: 15px;
      box-shadow: 0 5px 14px #625ee655;
      cursor: pointer
    }

    .links {
      display: flex;
      justify-content: center;
      gap: 16px
    }

    .links button {
      min-width: 200px;
      border: 1px solid #e2e8f0;
      background: #fff;
      border-radius: 8px;
      padding: 14px 20px;
      color: #475569;
      font-weight: bold;
      cursor: pointer
    }

    .notice {
      padding: 13px 16px;
      border-radius: 8px;
      margin-bottom: 16px;
      font-size: 14px;
      text-align: center;
      line-height: 1.4;
      font-weight: 600;
    }

    .error {
      background: #fff1f2;
      color: #be123c;
      border: 1px solid #fda4af;
    }

    .info {
      background: #eff6ff;
      color: #1d4ed8;
      border: 1px solid #93c5fd;
    }

    .success {
      background: #f0fdf4;
      color: #15803d;
      border: 1px solid #bbf7d0;
    }

    .input-error {
      border-color: #ef4444 !important;
      background-color: #fef2f2 !important;
      box-shadow: 0 0 0 3px rgba(239, 68, 68, 0.25) !important;
      animation: shake 0.35s ease-in-out;
    }

    @keyframes shake {
      0%, 100% { transform: translateX(0); }
      20%, 60% { transform: translateX(-6px); }
      40%, 80% { transform: translateX(6px); }
    }
  </style>
</head>

<body>
  <main class="card">
    <div class="icon">🔐</div>
    <h2>Sign in to your account</h2>
    <p>Securely access your Verify Portal account</p>
    <form class="form" id="form">
      <div id="notice"></div>
      <label>Username or Email Address</label>
      <div class="wrap">
        <span class="symbol">👤</span>
        <input id="email" class="field" type="text" placeholder="Enter your username or email" required>
      </div>
      <label>Password</label>
      <div class="wrap">
        <span class="symbol">🔒</span>
        <input id="password" class="field" type="password" placeholder="Enter your password" required>
        <button id="eye" class="eye" type="button">◉</button>
      </div>
      <button class="primary">Sign In to Portal →</button>
    </form>
    <div class="links">
      <button id="signup" type="button">✨ Create account</button>
      <button id="forgot" type="button">🔑 Forgot password?</button>
    </div>
  </main>
  <script>
    const post = (t, x) => parent.postMessage({ isStreamlitMessage: true, type: t, ...x }, '*');
    const emit = (action, data) => post('streamlit:setComponentValue', { value: { action, event_id: String(Date.now()), ...(data || {}) }, dataType: 'json' });

    const email = document.getElementById('email');
    const password = document.getElementById('password');
    const eye = document.getElementById('eye');

    [email, password].forEach(el => {
      el.addEventListener('input', () => el.classList.remove('input-error'));
    });

    document.getElementById('form').onsubmit = e => {
      e.preventDefault();
      emit('login', { email: email.value.trim(), password: password.value });
    };
    eye.onclick = () => {
      password.type = password.type === 'password' ? 'text' : 'password';
      eye.textContent = password.type === 'password' ? '◉' : '◌';
    };
    document.getElementById('signup').onclick = () => emit('signup');
    document.getElementById('forgot').onclick = () => emit('forgot');

    post('streamlit:componentReady', { apiVersion: 1 });
    post('streamlit:setFrameHeight', { height: 830 });

    let noticeTimer = null;
    window.addEventListener('message', e => {
      const data = e.data;
      if (data?.type !== 'streamlit:render') return;
      post('streamlit:setFrameHeight', { height: 830 });

      const n = document.getElementById('notice');
      const x = data.args?.notice;
      if (noticeTimer) clearTimeout(noticeTimer);
      email.classList.remove('input-error');
      password.classList.remove('input-error');
      if (!x) {
        n.innerHTML = '';
        return;
      }
      const kind = typeof x === 'string' ? 'error' : (x.kind || 'error');
      const text = typeof x === 'string' ? x : (x.text || '');
      n.innerHTML = '<div class="notice ' + kind + '">' + text + '</div>';
      n.style.transition = 'opacity 0.35s ease, transform 0.35s ease';
      n.style.opacity = '1';
      n.style.transform = 'translateY(0)';

      if (x.field === 'email') {
        email.classList.add('input-error');
        email.focus();
      } else if (x.field === 'password') {
        password.classList.add('input-error');
        password.focus();
      } else if (x.field === 'both') {
        email.classList.add('input-error');
        password.classList.add('input-error');
        email.focus();
      }

      noticeTimer = setTimeout(() => {
        n.style.opacity = '0';
        n.style.transform = 'translateY(-4px)';
        setTimeout(() => {
          n.innerHTML = '';
          n.style.opacity = '1';
          n.style.transform = 'none';
        }, 350);
      }, 4500);
    });
  </script>
</body>

</html>

In [ ]:
# @title Signup Component
%%writefile signup_component/index.html
<!doctype html>
<html>

<head>
  <meta charset="utf-8">
  <style>
    * {
      box-sizing: border-box
    }

    html,
    body {
      margin: 0;
      font-family: Arial, sans-serif;
      color: #17233b
    }

    body {
      display: grid;
      place-items: center;
      padding: 22px 22px 40px
    }

    .card {
      width: min(1120px, 95vw);
      background: #fff;
      border-radius: 16px;
      padding: 36px 60px 50px;
      box-shadow: 0 20px 50px #0003;
      text-align: center;
      position: relative;
      overflow: hidden;
    }

    .icon {
      width: 58px;
      height: 58px;
      margin: auto;
      display: grid;
      place-items: center;
      border-radius: 50%;
      background: linear-gradient(135deg, #6260e6, #8343ad);
      color: white;
      font-size: 25px;
      box-shadow: 0 8px 20px #625ee655
    }

    .subtitle {
      color: #64748b;
      margin: 7px 0 20px
    }

    .form {
      width: 590px;
      max-width: 100%;
      margin: auto;
      text-align: left
    }

    .field {
      margin: 11px 0
    }

    .field label {
      display: block;
      font-size: 13px;
      font-weight: 700;
      margin-bottom: 6px
    }

    .wrap {
      position: relative
    }

    .wrap input,
    .wrap select {
      width: 100%;
      height: 46px;
      border: 1.5px solid #cbd5e1;
      border-radius: 8px;
      padding: 0 43px;
      font-size: 14px;
      color: #17233b;
      background: #fff
    }

    .wrap select {
      padding-left: 14px
    }

    .mark {
      position: absolute;
      left: 14px;
      top: 13px;
      color: #64748b
    }

    .eye {
      position: absolute;
      right: 8px;
      top: 7px;
      border: 0;
      background: transparent;
      color: #475569;
      font-size: 19px;
      cursor: pointer
    }

    .meter {
      display: flex;
      gap: 4px;
      margin: 10px 0 5px
    }

    .meter i {
      height: 4px;
      flex: 1;
      background: #e2e8f0;
      border-radius: 3px
    }

    .meter.weak i:first-child {
      background: #ef4444
    }

    .meter.medium i:nth-child(-n+3) {
      background: #f59e0b
    }

    .meter.strong i {
      background: #22c55e
    }

    .rule {
      font-size: 13px;
      margin: 5px 0;
      color: #64748b
    }

    .ok {
      color: #16a34a
    }

    .bad {
      color: #dc2626
    }

    .security {
      border: 1px solid #e2e8f0;
      border-radius: 10px;
      padding: 10px 14px;
      margin: 15px 0;
      background: #f8fafc
    }

    .security h3 {
      font-size: 13px;
      margin: 0 0 3px
    }

    .security p {
      font-size: 12px;
      color: #64748b;
      margin: 0
    }

    .submit {
      width: 100%;
      height: 50px;
      border: 0;
      border-radius: 8px;
      background: linear-gradient(90deg, #6260e6, #8343ad);
      color: white;
      font-weight: 700;
      margin-top: 14px;
      cursor: pointer;
      display: flex;
      align-items: center;
      justify-content: center;
      gap: 8px;
      transition: all .2s ease;
    }

    .submit:disabled {
      opacity: .5;
      cursor: not-allowed
    }

    .link {
      display: block;
      border: 0;
      background: transparent;
      color: #625ee6;
      font-weight: 700;
      cursor: pointer;
      margin: 20px auto 0;
      font-size: 14px;
      text-decoration: underline;
      text-underline-offset: 3px
    }

    .notice {
      padding: 12px;
      border-radius: 8px;
      margin-bottom: 15px;
      font-size: 14px
    }

    .error {
      background: #fff1f2;
      color: #be123c;
      border: 1px solid #fda4af
    }

    .info {
      background: #eff6ff;
      color: #1d4ed8;
      border: 1px solid #93c5fd
    }

    .input-error {
      border-color: #ef4444 !important;
      background-color: #fef2f2 !important;
      box-shadow: 0 0 0 3px rgba(239, 68, 68, 0.25) !important;
      animation: shake 0.35s ease-in-out;
    }

    @keyframes shake {
      0%, 100% { transform: translateX(0); }
      20%, 60% { transform: translateX(-6px); }
      40%, 80% { transform: translateX(6px); }
    }

    .loading-overlay {
      position: absolute;
      inset: 0;
      background: rgba(255, 255, 255, 0.94);
      backdrop-filter: blur(6px);
      -webkit-backdrop-filter: blur(6px);
      display: flex;
      align-items: center;
      justify-content: center;
      z-index: 100;
      border-radius: 16px;
      animation: fadeIn 0.25s cubic-bezier(0.16, 1, 0.3, 1);
    }

    @keyframes fadeIn {
      from {
        opacity: 0;
        transform: scale(0.98)
      }

      to {
        opacity: 1;
        transform: scale(1)
      }
    }

    .loading-content {
      text-align: center;
      padding: 24px
    }

    .loading-spinner {
      width: 52px;
      height: 52px;
      margin: 0 auto 18px;
      border: 4px solid #ede9fe;
      border-top-color: #6260e6;
      border-right-color: #8343ad;
      border-radius: 50%;
      animation: spin 0.8s linear infinite;
      box-shadow: 0 4px 14px rgba(98, 96, 230, .25)
    }

    @keyframes spin {
      to {
        transform: rotate(360deg)
      }
    }

    .loading-title {
      font-size: 22px;
      font-weight: 700;
      color: #1e1b4b;
      margin-bottom: 8px;
      letter-spacing: -0.01em
    }

    .loading-desc {
      font-size: 14px;
      color: #64748b;
      max-width: 320px;
      margin: 0 auto;
      line-height: 1.5
    }

    .spinner-sm {
      display: inline-block;
      width: 18px;
      height: 18px;
      border: 2.5px solid rgba(255, 255, 255, 0.35);
      border-top-color: #fff;
      border-radius: 50%;
      animation: spin 0.8s linear infinite;
      vertical-align: middle
    }
  </style>
</head>

<body>
  <main class="card">
    <div id="loadingOverlay" class="loading-overlay" style="display:none;">
      <div class="loading-content">
        <div class="loading-spinner"></div>
        <div class="loading-title">Sending OTP...</div>
        <div class="loading-desc">Please wait while we send the verification code to your email.</div>
      </div>
    </div>
    <div class="icon">✦</div>
    <h2>Create your account</h2>
    <p class="subtitle">Register for secure Verify Portal access</p>
    <form class="form" id="form">
      <div id="notice"></div>
      <div class="field"><label>Full Name</label>
        <div class="wrap"><span class="mark">◉</span><input id="username" required placeholder="John Doe"></div>
      </div>
      <div class="field"><label>Email Address</label>
        <div class="wrap"><span class="mark">✉</span><input id="email" type="email" required
            placeholder="you@company.com"></div>
      </div>
      <div class="field"><label>Password</label>
        <div class="wrap"><span class="mark">🔒</span><input id="password" type="password" required
            placeholder="At least 8 characters"><button class="eye" type="button" data-eye="password">◉</button></div>
      </div>
      <div class="field"><label>Confirm Password</label>
        <div class="wrap"><span class="mark">🔒</span><input id="confirm" type="password" required
            placeholder="Re-enter password"><button class="eye" type="button" data-eye="confirm">◉</button></div>
      </div>
      <div id="meter" class="meter"><i></i><i></i><i></i><i></i><i></i></div>
      <div id="strength" class="rule">Password strength: —</div>
      <div id="rules">
        <div class="rule" data-rule="length">○ 8 or more characters</div>
        <div class="rule" data-rule="upper">○ At least 1 uppercase letter</div>
        <div class="rule" data-rule="lower">○ At least 1 lowercase letter</div>
        <div class="rule" data-rule="digit">○ At least 1 number</div>
        <div class="rule" data-rule="special">○ At least 1 special character</div>
      </div>
      <div id="match" class="rule"></div>
      <div class="security">
        <h3>Security Question</h3>
        <p>Required for secure account recovery if you forget your password.</p>
        <div class="field"><label>Choose a security question</label>
          <div class="wrap"><select id="question" required>
              <option value="" disabled selected>Select a security question</option>
              <option>What was the name of your first pet?</option>
              <option>What is your favourite city?</option>
              <option>What was your first school?</option>
              <option>In what city or town was your first job?</option>
              <option>What was the name of the street you grew up on?</option>
              <option>What was the name of your childhood best friend?</option>
              <option>What is your favorite book or movie?</option>
            </select></div>
        </div>
        <div class="field"><label>Security answer</label>
          <div class="wrap"><span class="mark">🔐</span><input id="answer" required
              placeholder="Enter your secret answer"></div>
        </div>
      </div>
      <button id="submit" class="submit" disabled>Create Account →</button>
    </form>
    <button id="signin" class="link" type="button">Already have an account? Sign in</button>
  </main>
  <script>
    const post = (t, x) => parent.postMessage({ isStreamlitMessage: true, type: t, ...x }, '*');
    const emit = (action, data) => post('streamlit:setComponentValue', { value: { action, event_id: String(Date.now()), ...(data || {}) }, dataType: 'json' });
    const $ = id => document.getElementById(id);

    const password = $('password'), confirm = $('confirm'), meter = $('meter'), submit = $('submit');

    function isSecurityAnswerValid(ans) {
      const trimmed = (ans || '').trim();
      if (trimmed.length < 3) return false;
      const stripped = trimmed.replace(/\s+/g, '');
      if (/^(.)\1+$/.test(stripped)) return false;
      const words = trimmed.split(/\s+/).filter(Boolean);
      if (words.length >= 3 && words.every(w => w.length === 1)) return false;
      return true;
    }

    function validate() {
      const p = password.value, c = confirm.value;
      const checks = { length: p.length >= 8, upper: /[A-Z]/.test(p), lower: /[a-z]/.test(p), digit: /\d/.test(p), special: /[!@#$%^&*(),.?":{}|<>]/.test(p) };
      const score = Object.values(checks).filter(Boolean).length;
      document.querySelectorAll('[data-rule]').forEach(x => {
        let ok = checks[x.dataset.rule];
        x.className = 'rule ' + (ok ? 'ok' : '');
        x.textContent = (ok ? '● ' : '○ ') + x.textContent.slice(2);
      });
      meter.className = 'meter ' + (score <= 2 ? 'weak' : score <= 4 ? 'medium' : 'strong');
      $('strength').textContent = 'Password strength: ' + (score === 0 ? '—' : score <= 2 ? 'Weak' : score <= 4 ? 'Moderate' : 'Strong');
      $('match').textContent = c ? (p === c ? '● Passwords match' : '○ Passwords do not match') : '';
      $('match').className = 'rule ' + (c && p === c ? 'ok' : 'bad');
      submit.disabled = !(Object.values(checks).every(Boolean) && p === c && $('username').value.trim() && $('email').value && Boolean($('question').value.trim()) && isSecurityAnswerValid($('answer').value));
    }

    function showLoading() {
      $('loadingOverlay').style.display = 'flex';
      submit.disabled = true;
      submit.innerHTML = '<span class="spinner-sm"></span> Sending OTP...';
    }

    function hideLoading() {
      $('loadingOverlay').style.display = 'none';
      submit.innerHTML = 'Create Account →';
      validate();
    }

    ['input', 'change'].forEach(e => document.querySelectorAll('input,select').forEach(x => {
      x.addEventListener(e, function() {
        this.classList.remove('input-error');
        validate();
      });
    }));
    document.querySelectorAll('[data-eye]').forEach(b => b.onclick = () => {
      const x = $(b.dataset.eye);
      x.type = x.type === 'password' ? 'text' : 'password';
      b.textContent = x.type === 'password' ? '◉' : '◌';
    });
    $('form').onsubmit = e => {
      e.preventDefault();
      showLoading();
      emit('signup', { username: $('username').value, email: $('email').value, password: password.value, confirm: confirm.value, question: $('question').value, answer: $('answer').value });
    };
    $('signin').onclick = () => emit('signin');

    function autoHeight() {
      const h = document.body.scrollHeight + 40;
      post('streamlit:setFrameHeight', { height: h });
    }

    post('streamlit:componentReady', { apiVersion: 1 });
    setTimeout(autoHeight, 100);

    let noticeTimer = null;
    window.addEventListener('message', e => {
      if (e.data?.type === 'streamlit:render') {
        setTimeout(autoHeight, 100);
        const notice = e.data.args?.notice;
        const n = $('notice');
        if (noticeTimer) clearTimeout(noticeTimer);
        document.querySelectorAll('.input-error').forEach(el => el.classList.remove('input-error'));
        if (notice && notice.text) {
          hideLoading();
          n.innerHTML = '<div class="notice ' + (notice.kind === 'error' ? 'error' : 'info') + '">' + notice.text + '</div>';
          n.style.transition = 'opacity 0.35s ease, transform 0.35s ease';
          n.style.opacity = '1';
          n.style.transform = 'translateY(0)';

          if (notice.field) {
            const el = $(notice.field);
            if (el) {
              el.classList.add('input-error');
              el.focus();
            }
          }

          noticeTimer = setTimeout(() => {
            n.style.opacity = '0';
            n.style.transform = 'translateY(-4px)';
            setTimeout(() => {
              n.innerHTML = '';
              n.style.opacity = '1';
              n.style.transform = 'none';
              autoHeight();
            }, 350);
          }, 4500);
        } else if (n) {
          n.innerHTML = '';
        }
      }
    });

    validate();
  </script>
</body>

</html>

In [ ]:
# @title Signup OTP Component
%%writefile signup_otp_component/index.html
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <style>
    * { box-sizing: border-box; }
    html, body { margin: 0; width: 100%; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Arial, sans-serif; }
    body { display: flex; justify-content: center; align-items: center; padding: 20px; background: transparent; }
    .card { width: min(1180px, 100%); padding: 45px 75px 50px; background: #fff; border-radius: 14px; box-shadow: 0 20px 50px rgba(0,0,0,.18); text-align: center; color: #1e293b; }
    .stepper { display: flex; align-items: center; justify-content: center; max-width: 320px; margin: 0 auto 48px; }
    .step { min-width: 60px; display: flex; flex-direction: column; align-items: center; }
    .circle { width: 32px; height: 32px; border-radius: 50%; display: grid; place-items: center; font-size: 13px; font-weight: 700; }
    .step-done .circle { background: #00b982; color: #fff; }
    .step-active .circle { background: #6260e6; color: #fff; box-shadow: 0 0 0 4px #e8eaff; }
    .step-pending .circle { background: #f1f5f9; color: #94a3b8; }
    .step-title { font-size: 11px; font-weight: 600; margin-top: 6px; white-space: nowrap; }
    .step-done .step-title { color: #00b982 }
    .step-active .step-title { color: #6260e6 }
    .step-pending .step-title { color: #94a3b8 }
    .line { height: 2px; flex: 1; margin: 0 8px 18px; border-radius: 2px; }
    .line-done { background: #00b982 }
    .line-inactive { background: #e2e8f0 }
    .icon { width: 58px; height: 58px; border-radius: 50%; margin: 0 auto 22px; display: grid; place-items: center; background: linear-gradient(135deg, #6260e6, #8343ad); box-shadow: 0 8px 20px rgba(98,96,230,.35); }
    h2 { margin: 0 0 14px; font-size: 24px; }
    .subtext { margin: 0; color: #64748b; font-size: 13px; line-height: 1.5; }
    .subtext strong { color: #6260e6; font-weight: 600; }
    .notice { display: none; max-width: 428px; margin: 14px auto 0; padding: 10px 12px; border-radius: 8px; font-size: 13px; font-weight: 600; }
    .notice.success { display: block; background: #ecfdf5; border: 1px solid #86efac; color: #047857; }
    .notice.error { display: block; background: #fff1f2; border: 1px solid #fda4af; color: #be123c; }
    .enter-label { margin: 26px 0 14px; color: #334155; font-size: 13px; font-weight: 600; }
    .otp-container { display: flex; justify-content: center; gap: 12px; margin-bottom: 20px; }
    .otp { width: 52px; height: 62px; border: 1.8px solid #6260e6; border-radius: 8px; background: #f0effe; outline: none; text-align: center; font-size: 22px; font-weight: 700; color: #1e293b; }
    .otp:focus { border-color: #4f46e5; background: #fff; }
    .spam-note { color: #94a3b8; font-size: 12px; margin-bottom: 22px; }
    .resend { font-size: 13px; color: #64748b; margin-bottom: 30px; }
    .link { border: 0; background: none; padding: 0; color: #6260e6; font: inherit; font-weight: 600; cursor: pointer; }
    .link:hover { text-decoration: underline; }
    .link:disabled { color: #94a3b8; cursor: not-allowed; text-decoration: none; }
    .verify { display: flex; align-items: center; justify-content: center; gap: 8px; width: 100%; max-width: 428px; height: 50px; margin: 0 auto; color: #fff; border: 0; border-radius: 8px; cursor: pointer; background: linear-gradient(90deg, #6260e6, #8343ad); box-shadow: 0 4px 14px rgba(96,93,236,.3); font-size: 14px; font-weight: 600; }
    .verify:disabled { opacity: .5; cursor: not-allowed; }
    .input-error {
      border-color: #ef4444 !important;
      background-color: #fef2f2 !important;
      box-shadow: 0 0 0 3px rgba(239, 68, 68, 0.25) !important;
      animation: shake 0.35s ease-in-out;
    }
    @keyframes shake {
      0%, 100% { transform: translateX(0); }
      20%, 60% { transform: translateX(-6px); }
      40%, 80% { transform: translateX(6px); }
    }
    @media(max-width:620px) {
      body { padding: 10px; }
      .card { padding: 28px 18px; }
      .otp-container { gap: 6px; }
      .otp { width: 42px; height: 52px; }
    }
  </style>
</head>
<body>
  <main class="card">
    <div class="stepper">
      <div class="step step-done"><div class="circle">✓</div><div class="step-title">Account</div></div>
      <div class="line line-done"></div>
      <div class="step step-active"><div class="circle">2</div><div class="step-title">Verify Email</div></div>
      <div class="line line-inactive"></div>
      <div class="step step-pending"><div class="circle">3</div><div class="step-title">Complete</div></div>
    </div>
    <div class="icon">
      <svg width="28" height="28" viewBox="0 0 24 24" fill="none" stroke="white" stroke-width="2" stroke-linecap="round" stroke-linejoin="round">
        <rect x="3" y="5" width="18" height="14" rx="2"/>
        <path d="m3 7 9 6 9-6"/>
      </svg>
    </div>
    <h2>Verify Your Email</h2>
    <p class="subtext">We sent a 6-digit verification code to <strong id="email">example@email.com</strong><br>Enter the code below to verify your account</p>
    <div id="notice" class="notice" role="status"></div>
    <div class="enter-label">Enter Verification Code</div>
    <form id="form">
      <div class="otp-container">
        <input class="otp" inputmode="numeric" maxlength="1" autofocus>
        <input class="otp" inputmode="numeric" maxlength="1">
        <input class="otp" inputmode="numeric" maxlength="1">
        <input class="otp" inputmode="numeric" maxlength="1">
        <input class="otp" inputmode="numeric" maxlength="1">
        <input class="otp" inputmode="numeric" maxlength="1">
      </div>
      <div class="spam-note">Check your spam folder if you don't see the email</div>
      <div class="resend">Didn't receive the code? <button class="link" type="button" id="resend" disabled>↻&nbsp; Resend Code (<span id="timer">30</span>s)</button></div>
      <button class="verify" type="submit">Verify &amp; Create Account</button>
    </form>
  </main>
  <script>
    const inputs = [...document.querySelectorAll('.otp')];
    const post = (type, extras = {}) => parent.postMessage({isStreamlitMessage: true, type, ...extras}, '*');
    const emit = (action, otp_code = '') => post('streamlit:setComponentValue', {value: {action, otp_code, event_id: `${Date.now()}-${Math.random()}`}, dataType: 'json'});

    document.getElementById('form').appendChild(document.getElementById('notice'));

    function autoHeight() {
      const h = document.body.scrollHeight + 40;
      post('streamlit:setFrameHeight', {height: h});
    }
    post('streamlit:componentReady', {apiVersion: 1});
    setTimeout(autoHeight, 100);

    const resend = document.getElementById('resend');
    let countdownTimer = null;

    function startResendCountdown(seconds = 30) {
      if (countdownTimer) {
        clearInterval(countdownTimer);
      }
      let remaining = seconds;
      resend.disabled = true;
      resend.innerHTML = `↻&nbsp; Resend Code (<span id="timer">${remaining}</span>s)`;

      countdownTimer = setInterval(() => {
        remaining -= 1;
        const timerEl = document.getElementById('timer');
        if (remaining <= 0) {
          clearInterval(countdownTimer);
          countdownTimer = null;
          resend.disabled = false;
          resend.textContent = '↻  Resend Code';
        } else if (timerEl) {
          timerEl.textContent = remaining;
        }
      }, 1000);
    }

    let noticeTimer = null;
    function setNotice(text, kind = 'info') {
      const noticeEl = document.getElementById('notice');
      if (noticeTimer) clearTimeout(noticeTimer);
      if (!text) {
        noticeEl.textContent = '';
        noticeEl.className = 'notice';
        noticeEl.style.opacity = '1';
        return;
      }
      noticeEl.textContent = text;
      noticeEl.className = `notice ${kind}`;
      noticeEl.style.transition = 'opacity 0.35s ease, transform 0.35s ease';
      noticeEl.style.opacity = '1';
      noticeEl.style.transform = 'translateY(0)';
      setTimeout(autoHeight, 50);

      noticeTimer = setTimeout(() => {
        noticeEl.style.opacity = '0';
        noticeEl.style.transform = 'translateY(-4px)';
        setTimeout(() => {
          noticeEl.textContent = '';
          noticeEl.className = 'notice';
          noticeEl.style.opacity = '1';
          noticeEl.style.transform = 'none';
          autoHeight();
        }, 350);
      }, 4500);
    }

    startResendCountdown(30);

    window.addEventListener('message', event => {
      const data = event.data;
      if (data?.type !== 'streamlit:render') return;
      setTimeout(autoHeight, 100);
      document.getElementById('email').textContent = data.args?.email || 'example@email.com';
      const notice = data.args?.notice;
      setNotice(notice?.text, notice?.kind || 'info');

      if (notice?.kind === 'error') {
        inputs.forEach(inp => inp.classList.add('input-error'));
        const isLocked = (notice.text || '').toLowerCase().includes('invalidated') || (notice.text || '').toLowerCase().includes('too many');
        if (isLocked) {
          inputs.forEach(inp => { inp.disabled = true; inp.value = ''; });
          const verifyBtn = document.querySelector('.verify');
          if (verifyBtn) verifyBtn.disabled = true;
          if (countdownTimer) {
            clearInterval(countdownTimer);
            countdownTimer = null;
          }
          resend.disabled = false;
          resend.innerHTML = '↻&nbsp; <strong>Request New Code</strong>';
          resend.style.color = '#ef4444';
        } else {
          inputs.forEach(inp => inp.value = '');
          if (inputs[0]) inputs[0].focus();
        }
      } else if (notice?.kind === 'success' && !countdownTimer) {
        startResendCountdown(30);
      }
    });

    inputs.forEach((input, index) => {
      input.addEventListener('input', () => {
        inputs.forEach(inp => inp.classList.remove('input-error'));
        input.value = input.value.replace(/\D/g, '').slice(0, 1);
        if (input.value && inputs[index + 1]) inputs[index + 1].focus();
      });
      input.addEventListener('keydown', e => {
        if (e.key === 'Backspace' && !input.value && inputs[index - 1]) inputs[index - 1].focus();
      });
      input.addEventListener('paste', e => {
        e.preventDefault();
        inputs.forEach(inp => inp.classList.remove('input-error'));
        const chars = (e.clipboardData.getData('text').match(/\d/g) || []).slice(0, 6);
        chars.forEach((char, i) => inputs[i].value = char);
        (inputs[Math.min(chars.length, 5)]).focus();
      });
    });

    document.getElementById('form').addEventListener('submit', e => { e.preventDefault(); emit('verify_otp', inputs.map(x => x.value).join('')); });
    resend.onclick = () => {
      inputs.forEach(inp => {
        inp.disabled = false;
        inp.value = '';
        inp.classList.remove('input-error');
      });
      const verifyBtn = document.querySelector('.verify');
      if (verifyBtn) verifyBtn.disabled = false;
      resend.style.color = '';
      if (inputs[0]) inputs[0].focus();
      startResendCountdown(30);
      emit('resend_otp');
    };
  </script>
</body>
</html>


In [ ]:
# @title Signup Complete Component
%%writefile signup_complete_component/index.html
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <style>
    * { box-sizing: border-box; }
    html, body { margin: 0; width: 100%; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Arial, sans-serif; }
    body { display: flex; justify-content: center; align-items: center; padding: 20px; background: transparent; }
    .card { width: min(1180px, 100%); padding: 50px 75px 55px; background: #fff; border-radius: 14px; box-shadow: 0 20px 50px rgba(0,0,0,.18); text-align: center; color: #1e293b; }
    .stepper { display: flex; align-items: center; justify-content: center; max-width: 320px; margin: 0 auto 48px; }
    .step { min-width: 60px; display: flex; flex-direction: column; align-items: center; }
    .circle { width: 32px; height: 32px; border-radius: 50%; display: grid; place-items: center; font-size: 13px; font-weight: 700; }
    .step-done .circle { background: #00b982; color: #fff; }
    .step-title { font-size: 11px; font-weight: 600; margin-top: 6px; white-space: nowrap; }
    .step-done .step-title { color: #00b982; }
    .line { height: 2px; flex: 1; margin: 0 8px 18px; border-radius: 2px; background: #00b982; }

    .success-icon {
      width: 90px; height: 90px; border-radius: 50%; margin: 10px auto 28px;
      display: grid; place-items: center;
      background: linear-gradient(135deg, #00b982, #059669);
      box-shadow: 0 10px 30px rgba(0,185,130,.35);
      animation: scaleIn .5s cubic-bezier(.175,.885,.32,1.275);
    }
    .success-icon svg { animation: drawCheck .6s ease .3s both; }
    @keyframes scaleIn { 0% { transform: scale(0); opacity: 0; } 100% { transform: scale(1); opacity: 1; } }
    @keyframes drawCheck { 0% { stroke-dashoffset: 30; opacity: 0; } 100% { stroke-dashoffset: 0; opacity: 1; } }

    h2 { margin: 0 0 12px; font-size: 28px; color: #1e293b; }
    .subtext { margin: 0 0 8px; color: #64748b; font-size: 14px; line-height: 1.6; }
    .email-highlight { color: #6260e6; font-weight: 700; }

    .info-box {
      max-width: 460px; margin: 28px auto; padding: 18px 22px;
      background: #ecfdf5; border: 1px solid #86efac; border-radius: 12px;
      text-align: left; font-size: 13px; color: #047857; line-height: 1.7;
    }
    .info-box strong { color: #065f46; }

    .signin-btn {
      display: inline-flex; align-items: center; justify-content: center; gap: 8px;
      width: 100%; max-width: 400px; height: 54px; margin: 20px auto 0;
      color: #fff; border: 0; border-radius: 10px; cursor: pointer;
      background: linear-gradient(90deg, #6260e6, #8343ad);
      box-shadow: 0 6px 18px rgba(96,93,236,.35);
      font-size: 15px; font-weight: 700; letter-spacing: .02em;
      transition: transform .2s, box-shadow .2s;
    }
    .signin-btn:hover { transform: translateY(-2px); box-shadow: 0 10px 24px rgba(96,93,236,.45); }

    .countdown { margin-top: 18px; font-size: 13px; color: #94a3b8; }
    .countdown span { color: #6260e6; font-weight: 700; }

    @media(max-width: 620px) {
      body { padding: 10px; }
      .card { padding: 32px 20px; }
    }
  </style>
</head>
<body>
  <main class="card">
    <div class="stepper">
      <div class="step step-done"><div class="circle">✓</div><div class="step-title">Account</div></div>
      <div class="line"></div>
      <div class="step step-done"><div class="circle">✓</div><div class="step-title">Verify Email</div></div>
      <div class="line"></div>
      <div class="step step-done"><div class="circle">✓</div><div class="step-title">Complete</div></div>
    </div>

    <div class="success-icon">
      <svg width="44" height="44" viewBox="0 0 24 24" fill="none" stroke="white" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" style="stroke-dasharray:30;stroke-dashoffset:30;">
        <path d="M5 13l4 4L19 7"/>
      </svg>
    </div>

    <h2>Account Created Successfully! 🎉</h2>
    <p class="subtext">
      Your account for <span class="email-highlight" id="email">example@email.com</span> has been verified and created.
    </p>
    <p class="subtext">You can now sign in with your credentials.</p>

    <div class="info-box">
      <strong>✓ What's ready:</strong><br>
      • Email address verified<br>
      • Password securely hashed<br>
      • Account activated and ready to use
    </div>

    <button class="signin-btn" id="signin">Sign In to Your Account →</button>
    <div class="countdown">Redirecting to sign in in <span id="counter">5</span> seconds...</div>
  </main>

  <script>
    const post = (t, x) => parent.postMessage({isStreamlitMessage: true, type: t, ...x}, '*');
    const emit = (action) => post('streamlit:setComponentValue', {value: {action, event_id: `${Date.now()}-${Math.random()}`}, dataType: 'json'});

    function autoHeight() {
      const h = document.body.scrollHeight + 40;
      post('streamlit:setFrameHeight', {height: h});
    }
    post('streamlit:componentReady', {apiVersion: 1});
    setTimeout(autoHeight, 100);

    window.addEventListener('message', e => {
      if (e.data?.type !== 'streamlit:render') return;
      setTimeout(autoHeight, 100);
      const email = e.data.args?.email;
      if (email) document.getElementById('email').textContent = email;
    });

    document.getElementById('signin').onclick = () => emit('go_login');

    // Auto redirect countdown
    let secs = 5;
    const counterEl = document.getElementById('counter');
    const timer = setInterval(() => {
      secs -= 1;
      counterEl.textContent = secs;
      if (secs <= 0) {
        clearInterval(timer);
        emit('go_login');
      }
    }, 1000);
  </script>
</body>
</html>


In [ ]:
# @title OTP Verification Component
%%writefile otp_verification_component/index.html
<!doctype html>
<html lang="en">

<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <style>
    * {
      box-sizing: border-box;
    }

    html,
    body {
      margin: 0;
      width: 100%;
      min-height: 750px;
      font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Arial, sans-serif;
    }

    body {
      display: flex;
      justify-content: center;
      align-items: center;
      padding: 20px;
      background: transparent;
    }

    .card {
      width: min(1180px, 100%);
      min-height: 790px;
      padding: 45px 75px;
      background: #fff;
      border-radius: 14px;
      box-shadow: 0 20px 50px rgba(0, 0, 0, .18);
      text-align: center;
      color: #1e293b;
    }

    .stepper {
      display: flex;
      align-items: center;
      justify-content: center;
      max-width: 320px;
      margin: 0 auto 48px;
    }

    .step {
      min-width: 60px;
      display: flex;
      flex-direction: column;
      align-items: center;
    }

    .circle {
      width: 32px;
      height: 32px;
      border-radius: 50%;
      display: grid;
      place-items: center;
      font-size: 13px;
      font-weight: 700;
    }

    .step-1 .circle {
      background: #00b982;
      color: #fff;
    }

    .step-2 .circle {
      background: #6260e6;
      color: #fff;
      box-shadow: 0 0 0 4px #e8eaff;
    }

    .step-3 .circle {
      background: #f1f5f9;
      color: #94a3b8;
    }

    .step-title {
      font-size: 11px;
      font-weight: 600;
      margin-top: 6px;
      white-space: nowrap;
    }

    .step-1 .step-title {
      color: #00b982
    }

    .step-2 .step-title {
      color: #6260e6
    }

    .step-3 .step-title {
      color: #94a3b8
    }

    .line {
      height: 2px;
      flex: 1;
      margin: 0 8px 18px;
      border-radius: 2px;
    }

    .line-active {
      background: #00b982
    }

    .line-inactive {
      background: #e2e8f0
    }

    .icon {
      width: 58px;
      height: 58px;
      border-radius: 50%;
      margin: 0 auto 22px;
      display: grid;
      place-items: center;
      background: linear-gradient(135deg, #6260e6, #8343ad);
      box-shadow: 0 8px 20px rgba(98, 96, 230, .35);
    }

    h2 {
      margin: 0 0 14px;
      font-size: 24px;
    }

    .subtext {
      margin: 0;
      color: #64748b;
      font-size: 13px;
      line-height: 1.5;
    }

    .subtext strong {
      color: #6260e6;
      font-weight: 600;
    }

    .notice {
      display: none;
      max-width: 428px;
      margin: 14px auto 0;
      padding: 10px 12px;
      border-radius: 8px;
      font-size: 13px;
      font-weight: 600;
    }

    .notice.success {
      display: block;
      background: #ecfdf5;
      border: 1px solid #86efac;
      color: #047857;
    }

    .notice.error {
      display: block;
      background: #fff1f2;
      border: 1px solid #fda4af;
      color: #be123c;
    }

    .enter-label {
      margin: 26px 0 14px;
      color: #334155;
      font-size: 13px;
      font-weight: 600;
    }

    .otp-container {
      display: flex;
      justify-content: center;
      gap: 12px;
      margin-bottom: 20px;
    }

    .otp {
      width: 52px;
      height: 62px;
      border: 1.8px solid #00b982;
      border-radius: 8px;
      background: #e8faf4;
      outline: none;
      text-align: center;
      font-size: 22px;
      font-weight: 700;
      color: #1e293b;
    }

    .otp:focus {
      border-color: #008f64;
      background: #fff;
    }

    .spam-note {
      color: #94a3b8;
      font-size: 12px;
      margin-bottom: 22px
    }

    .resend {
      font-size: 13px;
      color: #64748b;
      margin-bottom: 30px
    }

    .link {
      border: 0;
      background: none;
      padding: 0;
      color: #6260e6;
      font: inherit;
      font-weight: 600;
      cursor: pointer
    }

    .link:hover {
      text-decoration: underline
    }

    .link:disabled {
      color: #94a3b8;
      cursor: not-allowed;
      text-decoration: none
    }

    .verify {
      display: flex;
      align-items: center;
      justify-content: center;
      gap: 8px;
      width: 100%;
      max-width: 428px;
      height: 50px;
      margin: 0 auto;
      color: #fff;
      border: 0;
      border-radius: 8px;
      cursor: pointer;
      background: linear-gradient(90deg, #6260e6, #8343ad);
      box-shadow: 0 4px 14px rgba(96, 93, 236, .3);
      font-size: 14px;
      font-weight: 600;
    }
    .verify:disabled { opacity: .5; cursor: not-allowed; }
    .input-error {
      border-color: #ef4444 !important;
      background-color: #fef2f2 !important;
      box-shadow: 0 0 0 3px rgba(239, 68, 68, 0.25) !important;
      animation: shake 0.35s ease-in-out;
    }
    @keyframes shake {
      0%, 100% { transform: translateX(0); }
      20%, 60% { transform: translateX(-6px); }
      40%, 80% { transform: translateX(6px); }
    }

    .divider {
      display: flex;
      align-items: center;
      gap: 16px;
      max-width: 428px;
      margin: 42px auto 28px;
      color: #94a3b8;
      font-size: 12px;
    }

    .divider:before,
    .divider:after {
      content: "";
      height: 1px;
      background: #e2e8f0;
      flex: 1
    }

    .trouble {
      display: flex;
      justify-content: center;
      gap: 16px;
      margin-bottom: 16px;
    }

    .pill {
      min-width: 200px;
      height: 46px;
      border: 1px solid #e2e8f0;
      border-radius: 8px;
      color: #475569;
      background: #fff;
      cursor: pointer;
      font-size: 13px;
    }

    .pill:hover {
      background: #f8fafc;
      border-color: #cbd5e1
    }

    .signin {
      font-size: 13px;
      color: #64748b
    }

    @media(max-width:620px) {
      body {
        padding: 10px
      }

      .card {
        padding: 28px 18px
      }

      .otp-container {
        gap: 6px
      }

      .otp {
        width: 42px;
        height: 52px
      }

      .trouble {
        gap: 8px
      }

      .pill {
        min-width: 0;
        flex: 1;
        padding: 6px
      }

      .stepper {
        margin-bottom: 32px
      }
    }
  </style>
</head>

<body>
  <style>
    html,
    body {
      min-height: 900px;
    }

    .card {
      min-height: 820px;
    }

    #notice {
      margin-top: 20px;
      margin-bottom: 0;
    }
  </style>
  <main class="card">
    <div class="stepper">
      <div class="step step-1">
        <div class="circle">1</div>
        <div class="step-title">Account</div>
      </div>
      <div class="line line-active"></div>
      <div class="step step-2">
        <div class="circle">2</div>
        <div class="step-title">Verify</div>
      </div>
      <div class="line line-inactive"></div>
      <div class="step step-3">
        <div class="circle">3</div>
        <div class="step-title">Complete</div>
      </div>
    </div>
    <div class="icon"><svg width="28" height="28" viewBox="0 0 24 24" fill="none" stroke="white" stroke-width="2"
        stroke-linecap="round" stroke-linejoin="round">
        <rect x="3" y="5" width="18" height="14" rx="2" />
        <path d="m3 7 9 6 9-6" />
        <path d="M9 3h6" />
      </svg></div>
    <h2>Verify</h2>
    <p class="subtext">We sent a verification code to <strong id="email">example@email.com</strong><br>Enter the 6-digit
      code to continue</p>
    <div id="notice" class="notice" role="status"></div>
    <div class="enter-label">Enter Code</div>
    <form id="form">
      <div class="otp-container"><input class="otp" inputmode="numeric" maxlength="1" autofocus><input class="otp"
          inputmode="numeric" maxlength="1"><input class="otp" inputmode="numeric" maxlength="1"><input class="otp"
          inputmode="numeric" maxlength="1"><input class="otp" inputmode="numeric" maxlength="1"><input class="otp"
          inputmode="numeric" maxlength="1"></div>
      <div class="spam-note">Check your spam folder if you don't see the email</div>
      <div class="resend">Didn't receive the code? <button class="link" type="button" id="resend" disabled>↻&nbsp;
          Resend Code (<span id="timer">30</span>s)</button></div><button class="verify" type="submit">● &nbsp;
        Verify</button>
    </form>
    <div class="divider">having trouble?</div>
    <div class="trouble"><button class="pill" id="change" type="button">♧&nbsp; Change Email Address</button><button
        class="pill" id="support" type="button">♧&nbsp; Contact Support</button></div>
    <div class="signin">Already verified? <button class="link" id="signin" type="button">Sign in to your
        account</button></div>
  </main>
  <script>
    const inputs = [...document.querySelectorAll('.otp')];
    const post = (type, extras = {}) => parent.postMessage({ isStreamlitMessage: true, type, ...extras }, '*');
    const emit = (action, otp_code = '') => post('streamlit:setComponentValue', { value: { action, otp_code, event_id: `${Date.now()}-${Math.random()}` }, dataType: 'json' });
    // Keep feedback with the action that produced it, below Verify.
    document.getElementById('form').appendChild(document.getElementById('notice'));
    post('streamlit:componentReady', { apiVersion: 1 }); post('streamlit:setFrameHeight', { height: 900 });
    const resend = document.getElementById('resend');
    let countdownTimer = null;

    function startResendCountdown(seconds = 30) {
      if (countdownTimer) {
        clearInterval(countdownTimer);
      }
      let remaining = seconds;
      resend.disabled = true;
      resend.innerHTML = `↻&nbsp; Resend Code (<span id="timer">${remaining}</span>s)`;

      countdownTimer = setInterval(() => {
        remaining -= 1;
        const timerEl = document.getElementById('timer');
        if (remaining <= 0) {
          clearInterval(countdownTimer);
          countdownTimer = null;
          resend.disabled = false;
          resend.textContent = '↻  Resend Code';
        } else if (timerEl) {
          timerEl.textContent = remaining;
        }
      }, 1000);
    }

    let noticeTimer = null;
    function setNotice(text, kind = 'info') {
      const noticeElement = document.getElementById('notice');
      if (noticeTimer) clearTimeout(noticeTimer);
      if (!text) {
        noticeElement.textContent = '';
        noticeElement.className = 'notice';
        noticeElement.style.opacity = '1';
        return;
      }
      noticeElement.textContent = text;
      noticeElement.className = `notice ${kind}`;
      noticeElement.style.transition = 'opacity 0.35s ease, transform 0.35s ease';
      noticeElement.style.opacity = '1';
      noticeElement.style.transform = 'translateY(0)';

      noticeTimer = setTimeout(() => {
        noticeElement.style.opacity = '0';
        noticeElement.style.transform = 'translateY(-4px)';
        setTimeout(() => {
          noticeElement.textContent = '';
          noticeElement.className = 'notice';
          noticeElement.style.opacity = '1';
          noticeElement.style.transform = 'none';
        }, 350);
      }, 4500);
    }

    startResendCountdown(30);

    window.addEventListener('message', event => {
      const data = event.data;
      if (data?.type !== 'streamlit:render') return;
      document.getElementById('email').textContent = data.args?.email || 'example@email.com';
      const notice = data.args?.notice;
      setNotice(notice?.text, notice?.kind || 'info');

      if (notice?.kind === 'error') {
        inputs.forEach(inp => inp.classList.add('input-error'));
        const isLocked = (notice.text || '').toLowerCase().includes('invalidated') || (notice.text || '').toLowerCase().includes('too many');
        if (isLocked) {
          inputs.forEach(inp => { inp.disabled = true; inp.value = ''; });
          const verifyBtn = document.querySelector('.verify');
          if (verifyBtn) verifyBtn.disabled = true;
          if (countdownTimer) {
            clearInterval(countdownTimer);
            countdownTimer = null;
          }
          resend.disabled = false;
          resend.innerHTML = '↻&nbsp; <strong>Request New Code</strong>';
          resend.style.color = '#ef4444';
        } else {
          inputs.forEach(inp => inp.value = '');
          if (inputs[0]) inputs[0].focus();
        }
      } else if (notice?.kind === 'success' && !countdownTimer) {
        startResendCountdown(30);
      }
    });

    inputs.forEach((input, index) => {
      input.addEventListener('input', () => {
        inputs.forEach(inp => inp.classList.remove('input-error'));
        input.value = input.value.replace(/\D/g, '').slice(0, 1);
        if (input.value && inputs[index + 1]) inputs[index + 1].focus();
      });
      input.addEventListener('keydown', e => {
        if (e.key === 'Backspace' && !input.value && inputs[index - 1]) inputs[index - 1].focus();
      });
      input.addEventListener('paste', e => {
        e.preventDefault();
        inputs.forEach(inp => inp.classList.remove('input-error'));
        const chars = (e.clipboardData.getData('text').match(/\d/g) || []).slice(0, 6);
        chars.forEach((char, i) => inputs[i].value = char);
        (inputs[Math.min(chars.length, 5)]).focus();
      });
    });

    document.getElementById('form').addEventListener('submit', e => { e.preventDefault(); emit('verify_otp', inputs.map(x => x.value).join('')); });
    resend.onclick = () => {
      inputs.forEach(inp => {
        inp.disabled = false;
        inp.value = '';
        inp.classList.remove('input-error');
      });
      const verifyBtn = document.querySelector('.verify');
      if (verifyBtn) verifyBtn.disabled = false;
      resend.style.color = '';
      if (inputs[0]) inputs[0].focus();
      startResendCountdown(30);
      emit('resend_otp');
    };
    document.getElementById('change').onclick = () => emit('change_email');
    document.getElementById('signin').onclick = () => emit('signin');
    document.getElementById('support').onclick = () => alert('Support Assistance: support@verifyportal.com');
  </script>
</body>

</html>

In [ ]:
# @title Reset Email Component
%%writefile reset_email_component/index.html
<!doctype html>
<html>

<head>
  <meta charset="utf-8">
  <style>
    * {
      box-sizing: border-box
    }

    html,
    body {
      margin: 0;
      min-height: 790px;
      font-family: Arial, sans-serif;
      color: #17233b
    }

    body {
      display: grid;
      place-items: center;
      padding: 20px
    }

    .card {
      width: min(1120px, 95vw);
      min-height: 720px;
      background: #fff;
      border-radius: 16px;
      padding: 48px 60px;
      text-align: center;
      box-shadow: 0 20px 50px #0003;
      position: relative;
      overflow: hidden;
    }

    .steps {
      display: flex;
      align-items: flex-start;
      justify-content: center;
      gap: 18px;
      margin-bottom: 56px
    }

    .step {
      width: 100px;
      font-size: 12px;
      color: #94a3b8;
      font-weight: 700
    }

    .circle {
      width: 40px;
      height: 40px;
      border-radius: 50%;
      margin: auto auto 8px;
      display: grid;
      place-items: center;
      background: #f1f5f9
    }

    .active .circle {
      background: #6260e6;
      color: #fff;
      box-shadow: 0 0 0 5px #e0e7ff
    }

    .active {
      color: #625ee6
    }

    .line {
      width: 68px;
      height: 2px;
      background: #dbe4ef;
      margin-top: 19px
    }

    .icon {
      width: 72px;
      height: 72px;
      border-radius: 50%;
      display: grid;
      place-items: center;
      margin: auto;
      background: linear-gradient(135deg, #6260e6, #8343ad);
      color: #fff;
      font-size: 31px;
      box-shadow: 0 10px 24px #625ee655
    }

    .card h1 {
      font-size: 29px;
      margin: 28px 0 12px
    }

    .card p {
      color: #64748b;
      line-height: 1.55
    }

    .form {
      width: 535px;
      max-width: 100%;
      margin: 32px auto 0;
      text-align: left
    }

    .form label {
      font-weight: bold;
      font-size: 14px;
      display: block;
      margin-bottom: 8px
    }

    .form input {
      width: 100%;
      height: 58px;
      border: 1.5px solid #cbd5e1;
      border-radius: 9px;
      padding: 0 18px;
      font-size: 16px
    }

    .submit,
    .alt {
      width: 100%;
      height: 58px;
      border: 0;
      border-radius: 9px;
      font-size: 16px;
      font-weight: bold;
      cursor: pointer
    }

    .submit {
      margin-top: 20px;
      background: linear-gradient(90deg, #6260e6, #8343ad);
      color: #fff;
      box-shadow: 0 10px 18px #625ee644;
      display: flex;
      align-items: center;
      justify-content: center;
      gap: 10px;
      transition: all 0.2s ease;
    }

    .submit:disabled {
      opacity: 0.8;
      cursor: not-allowed;
    }

    .or {
      display: flex;
      align-items: center;
      gap: 12px;
      color: #94a3b8;
      font-size: 13px;
      margin: 19px 0
    }

    .or:before,
    .or:after {
      content: "";
      height: 1px;
      background: #e2e8f0;
      flex: 1
    }

    .alt {
      height: 46px;
      border: 1px solid #cbd5e1;
      background: #fff;
      color: #625ee6
    }

    .notice {
      padding: 12px;
      border-radius: 8px;
      margin-bottom: 15px;
      font-size: 14px
    }

    .error {
      background: #fff1f2;
      color: #be123c;
      border: 1px solid #fda4af
    }

    .info {
      background: #eff6ff;
      color: #1d4ed8;
      border: 1px solid #93c5fd
    }

    .input-error {
      border-color: #ef4444 !important;
      background-color: #fef2f2 !important;
      box-shadow: 0 0 0 3px rgba(239, 68, 68, 0.25) !important;
      animation: shake 0.35s ease-in-out;
    }

    @keyframes shake {
      0%, 100% { transform: translateX(0); }
      20%, 60% { transform: translateX(-6px); }
      40%, 80% { transform: translateX(6px); }
    }

    .signin {
      border: 0;
      background: none;
      color: #625ee6;
      font-weight: bold;
      margin-top: 26px;
      cursor: pointer
    }

    .loading-overlay {
      position: absolute;
      inset: 0;
      background: rgba(255, 255, 255, 0.94);
      backdrop-filter: blur(6px);
      -webkit-backdrop-filter: blur(6px);
      display: flex;
      align-items: center;
      justify-content: center;
      z-index: 100;
      border-radius: 16px;
      animation: fadeIn 0.25s cubic-bezier(0.16, 1, 0.3, 1);
    }

    @keyframes fadeIn {
      from { opacity: 0; transform: scale(0.98); }
      to { opacity: 1; transform: scale(1); }
    }

    .loading-content {
      text-align: center;
      padding: 24px;
    }

    .loading-spinner {
      width: 52px;
      height: 52px;
      margin: 0 auto 18px;
      border: 4px solid #ede9fe;
      border-top-color: #6260e6;
      border-right-color: #8343ad;
      border-radius: 50%;
      animation: spin 0.8s linear infinite;
      box-shadow: 0 4px 14px rgba(98, 96, 230, 0.25);
    }

    @keyframes spin {
      to { transform: rotate(360deg); }
    }

    .loading-title {
      font-size: 22px;
      font-weight: 700;
      color: #1e1b4b;
      margin-bottom: 8px;
      letter-spacing: -0.01em;
    }

    .loading-subtitle {
      font-size: 14px;
      color: #64748b;
      max-width: 320px;
      margin: 0 auto;
      line-height: 1.5;
    }

    .spinner-sm {
      display: inline-block;
      width: 18px;
      height: 18px;
      border: 2.5px solid rgba(255, 255, 255, 0.35);
      border-top-color: #fff;
      border-radius: 50%;
      animation: spin 0.8s linear infinite;
      vertical-align: middle;
    }
  </style>
</head>

<body>
  <main class="card">
    <div id="loadingOverlay" class="loading-overlay" style="display:none;">
      <div class="loading-content">
        <div class="loading-spinner"></div>
        <div class="loading-title">Sending OTP...</div>
        <div class="loading-subtitle">Please wait while we send your 6-digit verification code.</div>
      </div>
    </div>
    <div class="steps">
      <div class="step active">
        <div class="circle">1</div>Account
      </div>
      <div class="line"></div>
      <div class="step">
        <div class="circle">2</div>Verify
      </div>
      <div class="line"></div>
      <div class="step">
        <div class="circle">3</div>Complete
      </div>
    </div>
    <div class="icon">✉</div>
    <h1>Reset Password</h1>
    <p>Enter your registered email address.<br>We'll send a 6-digit verification code to continue.</p>
    <form id="form" class="form">
      <div id="notice"></div>
      <label>Email Address or Username</label>
      <input id="email" type="text" placeholder="you@company.com or username" required>
      <button class="submit" id="submitBtn">Send Verification Code &nbsp; →</button>
      <div class="or">or</div>
      <button class="alt" id="security" type="button">🔐 Use security question instead</button>
    </form>
    <button class="signin" id="signin" type="button">Remembered your password? Sign in to your account</button>
  </main>
  <script>
    const post = (t, x) => parent.postMessage({ isStreamlitMessage: true, type: t, ...x }, '*');
    const emit = (action, extra) => post('streamlit:setComponentValue', { value: { action, email: document.getElementById('email').value, event_id: String(Date.now()), ...(extra || {}) }, dataType: 'json' });

    function showLoading() {
      const overlay = document.getElementById('loadingOverlay');
      if (overlay) overlay.style.display = 'flex';
      const btn = document.getElementById('submitBtn');
      if (btn) {
        btn.disabled = true;
        btn.innerHTML = '<span class="spinner-sm"></span> Sending OTP...';
      }
    }

    function hideLoading() {
      const overlay = document.getElementById('loadingOverlay');
      if (overlay) overlay.style.display = 'none';
      const btn = document.getElementById('submitBtn');
      if (btn) {
        btn.disabled = false;
        btn.innerHTML = 'Send Verification Code &nbsp; →';
      }
    }

    const emailInput = document.getElementById('email');
    emailInput.addEventListener('input', () => emailInput.classList.remove('input-error'));

    document.getElementById('form').onsubmit = e => {
      e.preventDefault();
      showLoading();
      emit('request_otp');
    };
    document.getElementById('security').onclick = () => emit('security_question');
    document.getElementById('signin').onclick = () => emit('signin');

    post('streamlit:componentReady', { apiVersion: 1 });
    post('streamlit:setFrameHeight', { height: 830 });

    let noticeTimer = null;
    window.addEventListener('message', e => {
      const data = e.data;
      if (data?.type !== 'streamlit:render') return;
      const n = document.getElementById('notice');
      const x = data.args?.notice;
      if (noticeTimer) clearTimeout(noticeTimer);
      emailInput.classList.remove('input-error');
      if (!x) {
        n.innerHTML = '';
        return;
      }
      hideLoading();
      n.innerHTML = '<div class="notice ' + (x.kind === 'error' ? 'error' : 'info') + '">' + x.text + '</div>';
      n.style.transition = 'opacity 0.35s ease, transform 0.35s ease';
      n.style.opacity = '1';
      n.style.transform = 'translateY(0)';

      if (x.kind === 'error' || x.field === 'email') {
        emailInput.classList.add('input-error');
        emailInput.focus();
      }

      noticeTimer = setTimeout(() => {
        n.style.opacity = '0';
        n.style.transform = 'translateY(-4px)';
        setTimeout(() => {
          n.innerHTML = '';
          n.style.opacity = '1';
          n.style.transform = 'none';
        }, 350);
      }, 4500);
    });
  </script>
</body>

</html>

In [ ]:
# @title Reset Password Component
%%writefile reset_password_component/index.html
<!doctype html>
<html>

<head>
    <meta charset="utf-8">
    <style>
        * {
            box-sizing: border-box
        }

        html,
        body {
            margin: 0;
            width: 100%;
            min-height: 790px;
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Arial, sans-serif
        }

        body {
            display: flex;
            align-items: center;
            justify-content: center;
            padding: 20px
        }

        .card {
            width: 100%;
            max-width: 860px;
            min-height: 730px;
            padding: 42px 75px;
            background: #fff;
            border-radius: 14px;
            box-shadow: 0 20px 50px #0003;
            text-align: center;
            color: #1e293b
        }

        .steps {
            display: flex;
            align-items: center;
            justify-content: center;
            gap: 8px;
            margin: 0 auto 35px;
            font-size: 11px;
            font-weight: 600
        }

        .step {
            min-width: 60px
        }

        .circle {
            width: 32px;
            height: 32px;
            border-radius: 50%;
            display: grid;
            place-items: center;
            margin: auto
        }

        .done .circle {
            background: #00b982;
            color: #fff
        }

        .current .circle {
            background: #6260e6;
            color: #fff;
            box-shadow: 0 0 0 4px #e8eaff
        }

        .line {
            width: 70px;
            height: 2px;
            background: #00b982
        }

        .done {
            color: #00b982
        }

        .current {
            color: #6260e6
        }

        .icon {
            width: 56px;
            height: 56px;
            border-radius: 50%;
            margin: 0 auto 17px;
            display: grid;
            place-items: center;
            background: linear-gradient(135deg, #6260e6, #8343ad);
            color: white;
            font-size: 26px;
            box-shadow: 0 8px 20px #6260e655
        }

        h2 {
            margin: 0 0 8px;
            font-size: 24px
        }

        .sub {
            margin: 0;
            color: #64748b;
            font-size: 13px
        }

        .form {
            max-width: 428px;
            margin: 26px auto 0;
            text-align: left
        }

        .field {
            margin-bottom: 15px
        }

        .field label {
            display: block;
            margin-bottom: 8px;
            font-size: 13px;
            font-weight: 600
        }

        .wrap {
            position: relative
        }

        .password {
            width: 100%;
            height: 49px;
            border: 1.5px solid #cbd5e1;
            border-radius: 8px;
            padding: 0 46px 0 44px;
            font-size: 15px;
            outline: none
        }

        .password:focus {
            border-color: #6260e6;
            box-shadow: 0 0 0 3px #e8eaff
        }

        .lock {
            position: absolute;
            left: 14px;
            top: 14px;
            color: #64748b;
            width: 20px;
            height: 20px
        }

        .eye {
            position: absolute;
            right: 11px;
            top: 10px;
            border: 0;
            background: none;
            color: #64748b;
            width: 28px;
            height: 28px;
            padding: 0;
            cursor: pointer
        }

        .eye svg,
        .lock svg {
            width: 20px;
            height: 20px
        }

        .meter {
            display: flex;
            gap: 4px;
            margin: -4px 0 7px
        }

        .meter i {
            height: 4px;
            flex: 1;
            border-radius: 3px;
            background: #e2e8f0
        }

        .meter i.on:nth-child(1) {
            background: #ef4444
        }

        .meter i.on:nth-child(2) {
            background: #f59e0b
        }

        .meter i.on:nth-child(3) {
            background: #eab308
        }

        .meter i.on:nth-child(4),
        .meter i.on:nth-child(5) {
            background: #22c55e
        }

        .strength {
            font-size: 12px;
            color: #64748b;
            margin: 0 0 10px
        }

        .rules {
            margin: 5px 0 20px;
            padding: 0;
            list-style: none
        }

        .rule {
            display: flex;
            gap: 8px;
            align-items: center;
            margin: 7px 0;
            color: #64748b;
            font-size: 13px
        }

        .check {
            display: grid;
            place-items: center;
            width: 19px;
            height: 19px;
            border-radius: 50%;
            background: #e2e8f0;
            color: #fff;
            font-size: 12px;
            font-weight: 800
        }

        .rule.ok .check {
            background: #22c55e
        }

        .match {
            font-size: 12px;
            height: 17px;
            margin-top: -8px;
            color: #e11d48
        }

        .match.ok {
            color: #16a34a
        }

        .notice {
            display: none;
            margin-bottom: 14px;
            padding: 10px 12px;
            border-radius: 8px;
            font-size: 13px;
            font-weight: 600
        }

        .notice.error {
            display: block;
            background: #fff1f2;
            border: 1px solid #fda4af;
            color: #be123c
        }

        .input-error {
            border-color: #ef4444 !important;
            background-color: #fef2f2 !important;
            box-shadow: 0 0 0 3px rgba(239, 68, 68, 0.25) !important;
            animation: shake 0.35s ease-in-out;
        }

        @keyframes shake {
            0%, 100% { transform: translateX(0); }
            20%, 60% { transform: translateX(-6px); }
            40%, 80% { transform: translateX(6px); }
        }

        .submit {
            width: 100%;
            height: 50px;
            border: 0;
            border-radius: 8px;
            color: #fff;
            cursor: pointer;
            background: linear-gradient(90deg, #6260e6, #8343ad);
            font-size: 14px;
            font-weight: 600
        }

        .submit:disabled {
            opacity: .48;
            cursor: not-allowed
        }

        .signin {
            margin-top: 20px;
            font-size: 13px;
            color: #64748b
        }

        .link {
            border: 0;
            background: none;
            padding: 0;
            color: #6260e6;
            font: inherit;
            font-weight: 600;
            cursor: pointer
        }
    </style>
</head>

<body>
    <main class="card">
        <div class="steps">
            <div class="step done">
                <div class="circle">✓</div>Account
            </div><i class="line"></i>
            <div class="step done">
                <div class="circle">✓</div>Verify
            </div><i class="line"></i>
            <div class="step current">
                <div class="circle">3</div>Complete
            </div>
        </div>
        <div class="icon">▣</div>
        <h2>Create New Password</h2>
        <p class="sub">Choose a strong password to secure your account.</p>
        <form id="form" class="form">
            <div id="notice" class="notice"></div>
            <div class="field"><label>New Password</label>
                <div class="wrap"><span class="lock"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor"
                            stroke-width="2">
                            <rect x="4" y="10" width="16" height="10" rx="2" />
                            <path d="M8 10V7a4 4 0 0 1 8 0v3" />
                        </svg></span><input id="password" class="password" type="password"
                        autocomplete="new-password"><button class="eye" type="button" data-target="password"
                        aria-label="Show password"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor"
                            stroke-width="2">
                            <path d="M2 12s3.5-6 10-6 10 6 10 6-3.5 6-10 6S2 12 2 12Z" />
                            <circle cx="12" cy="12" r="2.5" />
                        </svg></button></div>
            </div>
            <div class="field"><label>Confirm New Password</label>
                <div class="wrap"><span class="lock"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor"
                            stroke-width="2">
                            <rect x="4" y="10" width="16" height="10" rx="2" />
                            <path d="M8 10V7a4 4 0 0 1 8 0v3" />
                        </svg></span><input id="confirm" class="password" type="password"
                        autocomplete="new-password"><button class="eye" type="button" data-target="confirm"
                        aria-label="Show confirmation password"><svg viewBox="0 0 24 24" fill="none"
                            stroke="currentColor" stroke-width="2">
                            <path d="M2 12s3.5-6 10-6 10 6 10 6-3.5 6-10 6S2 12 2 12Z" />
                            <circle cx="12" cy="12" r="2.5" />
                        </svg></button></div>
            </div>
            <div class="meter" id="meter"><i></i><i></i><i></i><i></i><i></i></div>
            <p id="strength" class="strength">Password strength: —</p>
            <div id="match" class="match"></div>
            <ul class="rules">
                <li class="rule" data-rule="length"><span class="check">✓</span>8 or more characters</li>
                <li class="rule" data-rule="upper"><span class="check">✓</span>At least 1 uppercase letter</li>
                <li class="rule" data-rule="lower"><span class="check">✓</span>At least 1 lowercase letter</li>
                <li class="rule" data-rule="digit"><span class="check">✓</span>At least 1 number</li>
                <li class="rule" data-rule="special"><span class="check">✓</span>At least 1 special character</li>
            </ul><button id="submit" class="submit" type="submit" disabled>Change Password</button>
        </form>
        <div class="signin">Want to start over? <button id="signin" class="link" type="button">Sign in to your
                account</button></div>
    </main>
    <script>
        const p = document.querySelector('#password'), c = document.querySelector('#confirm'), b = document.querySelector('#submit'), m = document.querySelector('#match'), bars = [...document.querySelectorAll('#meter i')], s = document.querySelector('#strength'); const post = (t, x = {}) => parent.postMessage({ isStreamlitMessage: true, type: t, ...x }, '*'), emit = (action, x = {}) => post('streamlit:setComponentValue', { value: { action, event_id: `${Date.now()}-${Math.random()}`, ...x }, dataType: 'json' }), rules = { length: x => x.length >= 8, upper: x => /[A-Z]/.test(x), lower: x => /[a-z]/.test(x), digit: x => /\d/.test(x), special: x => /[!@#$%^&*(),.?":{}|<>]/.test(x) }; function validate() { let r = Object.entries(rules).map(([k, f]) => { let ok = f(p.value); document.querySelector(`[data-rule="${k}"]`).classList.toggle('ok', ok); return ok }), score = r.filter(Boolean).length; bars.forEach((x, i) => x.classList.toggle('on', i < score)); s.textContent = `Password strength: ${score === 0 ? '—' : score < 3 ? 'Weak' : score < 5 ? 'Moderate' : 'Strong'}`; let same = c.value && p.value === c.value; m.textContent = c.value ? (same ? '✓ Passwords match' : 'Passwords do not match') : ''; m.className = `match ${same ? 'ok' : ''}`; b.disabled = !(score === 5 && same) } [p, c].forEach(x => { x.addEventListener('input', () => { x.classList.remove('input-error'); validate(); }); }); document.querySelectorAll('.eye').forEach(x => x.onclick = () => { let i = document.getElementById(x.dataset.target); i.type = i.type === 'password' ? 'text' : 'password' }); document.querySelector('#form').onsubmit = e => { e.preventDefault(); emit('update_password', { new_password: p.value, confirm_password: c.value }) }; document.querySelector('#signin').onclick = () => emit('signin'); post('streamlit:componentReady', { apiVersion: 1 }); post('streamlit:setFrameHeight', { height: 790 }); window.addEventListener('message', e => { let n = e.data?.args?.notice, x = document.querySelector('#notice'); p.classList.remove('input-error'); c.classList.remove('input-error'); x.textContent = n?.text || ''; x.className = n?.text ? 'notice error' : 'notice'; if (n?.field === 'password') { p.classList.add('input-error'); p.focus(); } else if (n?.field === 'confirm') { c.classList.add('input-error'); c.focus(); } });
    </script>
</body>

</html>

In [ ]:
# @title Security Recovery Component
%%writefile security_recovery_component/index.html
<!doctype html><html><head><meta charset="utf-8"><style>
*{box-sizing:border-box}html,body{margin:0;min-height:760px;font-family:Arial,sans-serif;color:#17233b}body{display:grid;place-items:center;padding:20px}.card{width:min(1120px,95vw);min-height:680px;background:#fff;border-radius:16px;padding:48px 60px;text-align:center;box-shadow:0 20px 50px #0003}.steps{display:flex;align-items:flex-start;justify-content:center;gap:18px;margin-bottom:54px}.step{width:100px;font-size:12px;color:#94a3b8;font-weight:700}.circle{width:40px;height:40px;border-radius:50%;margin:auto auto 8px;display:grid;place-items:center;background:#f1f5f9}.done .circle{background:#00b982;color:#fff}.active .circle{background:#6260e6;color:#fff;box-shadow:0 0 0 5px #e0e7ff}.active{color:#625ee6}.done{color:#00b982}.line{width:68px;height:2px;background:#dbe4ef;margin-top:19px}.line-done{background:#00b982}.icon{width:72px;height:72px;border-radius:50%;display:grid;place-items:center;margin:auto;background:linear-gradient(135deg,#6260e6,#8343ad);color:#fff;font-size:31px;box-shadow:0 10px 24px #625ee655}.card h1{font-size:29px;margin:27px 0 10px}.description{color:#64748b;margin:0}.form{width:535px;max-width:100%;margin:30px auto 0;text-align:left}.question{padding:15px;border:1px solid #dbe4ef;border-radius:9px;background:#f8fafc;color:#334155;line-height:1.45;margin:10px 0 18px;font-weight:700}.form label{font-weight:bold;font-size:14px;display:block;margin-bottom:8px}.form input{width:100%;height:58px;border:1.5px solid #cbd5e1;border-radius:9px;padding:0 18px;font-size:16px}.submit{width:100%;height:58px;border:0;border-radius:9px;font-size:16px;font-weight:bold;cursor:pointer;margin-top:20px;background:linear-gradient(90deg,#6260e6,#8343ad);color:#fff;box-shadow:0 10px 18px #625ee644}.notice{padding:12px;border-radius:8px;margin-bottom:15px;font-size:14px;background:#fff1f2;color:#be123c;border:1px solid #fda4af}.bottom{margin-top:23px;display:flex;justify-content:center;gap:25px}.bottom button{border:0;background:none;color:#625ee6;font-weight:bold;cursor:pointer}
.input-error{border-color:#ef4444!important;background-color:#fef2f2!important;box-shadow:0 0 0 3px rgba(239,68,68,0.25)!important;animation:shake 0.35s ease-in-out}@keyframes shake{0%,100%{transform:translateX(0)}20%,60%{transform:translateX(-6px)}40%,80%{transform:translateX(6px)}}
</style></head><body>
<main class="card">
  <div class="steps">
    <div class="step done"><div class="circle">✓</div>Account</div>
    <div class="line line-done"></div>
    <div class="step active"><div class="circle">2</div>Verify</div>
    <div class="line"></div>
    <div class="step"><div class="circle">3</div>Complete</div>
  </div>
  <div class="icon">🔐</div>
  <h1>Answer security question</h1>
  <p class="description">Verify your identity to reset the password for <strong id="email"></strong>.</p>
  <form class="form" id="form">
    <div id="notice"></div>
    <label>Your security question</label>
    <div class="question" id="question"></div>
    <label>Security answer</label>
    <input id="answer" autocomplete="off" required placeholder="Enter your answer">
    <button class="submit">Continue to new password &nbsp; →</button>
  </form>
  <div class="bottom">
    <button id="back" type="button">← Use email verification</button>
    <button id="signin" type="button">Sign in instead</button>
  </div>
</main>
<script>
  const post = (t, x) => parent.postMessage({isStreamlitMessage: true, type: t, ...x}, '*');
  const emit = (action, data) => post('streamlit:setComponentValue', {value: {action, event_id: String(Date.now()), ...(data||{})}, dataType: 'json'});

  const ansInput = document.getElementById('answer');
  const submitBtn = document.querySelector('.submit');
  const noticeEl = document.getElementById('notice');

  ansInput.addEventListener('input', () => {
    ansInput.classList.remove('input-error');
    if (!ansInput.disabled && noticeEl) {
      noticeEl.innerHTML = '';
    }
  });

  function isAnswerValid(ans) {
    const t = (ans || '').trim();
    if (t.length < 3) return false;
    const stripped = t.replace(/\s+/g, '');
    if (/^(.)\1+$/.test(stripped)) return false;
    const words = t.split(/\s+/).filter(Boolean);
    if (words.length >= 3 && words.every(w => w.length === 1)) return false;
    return true;
  }

  document.getElementById('form').onsubmit = e => {
    e.preventDefault();
    if (ansInput.disabled) return;
    const val = ansInput.value.trim();
    if (!isAnswerValid(val)) {
      ansInput.classList.add('input-error');
      ansInput.focus();
      noticeEl.innerHTML = '<div class="notice">Security answer is invalid. Please enter a valid word or phrase (not spaced single letters).</div>';
      return;
    }
    emit('verify_answer', {answer: val});
  };
  document.getElementById('back').onclick = () => emit('back');
  document.getElementById('signin').onclick = () => emit('signin');

  post('streamlit:componentReady', {apiVersion: 1});
  post('streamlit:setFrameHeight', {height: 800});

  window.addEventListener('message', e => {
    const data = e.data;
    if (data?.type !== 'streamlit:render') return;
    const a = data.args || {};
    document.getElementById('email').textContent = a.email || '';
    document.getElementById('question').textContent = a.question || 'Security question';
    
    if (a.notice && a.notice.text) {
      noticeEl.innerHTML = '<div class="notice">' + a.notice.text + '</div>';
      noticeEl.style.opacity = '1';
      ansInput.classList.add('input-error');

      if (a.notice.locked) {
        ansInput.disabled = true;
        if (submitBtn) {
          submitBtn.disabled = true;
          submitBtn.style.opacity = '0.5';
          submitBtn.style.cursor = 'not-allowed';
        }
      } else {
        ansInput.disabled = false;
        if (submitBtn) {
          submitBtn.disabled = false;
          submitBtn.style.opacity = '1';
          submitBtn.style.cursor = 'pointer';
        }
        ansInput.focus();
        ansInput.select();
      }
    } else {
      noticeEl.innerHTML = '';
      ansInput.classList.remove('input-error');
    }
  });
</script>
</body></html>


In [ ]:
# @title Configuration
%%writefile config.py
import os
import streamlit as st
from dotenv import load_dotenv

# Load environment variables from .env file if present
load_dotenv()

APP_NAME = "Verify Portal"
DB_PATH = os.getenv("DB_PATH", "portal_secure.db")

# Read from Colab Secrets, environment variables, or Streamlit secrets
def get_secret(key, default=None):
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val is not None:
            return str(val)
    except Exception:
        pass
    val = os.getenv(key)
    if val:
        return val
    try:
        if hasattr(st, "secrets") and key in st.secrets:
            return str(st.secrets[key])
    except Exception:
        pass
    return default

JWT_SECRET = get_secret("JWT_SECRET", "infosys-batch4-milestone1-jwt-key-2026")
SENDER_EMAIL = get_secret("EMAIL_ADDRESS") or get_secret("EMAIL_ID", "")
EMAIL_PASSWORD = get_secret("EMAIL_PASSWORD", "")

# Admin credentials defined directly in code per Step 11
ADMIN_USERNAME = get_secret("ADMIN_USERNAME", "admin")
ADMIN_EMAIL = get_secret("ADMIN_EMAIL", "admin@example.com")
ADMIN_PASSWORD = get_secret("ADMIN_PASSWORD", "Admin@123456")

OTP_EXPIRY_MINUTES = 5
MAX_LOGIN_ATTEMPTS = 3
LOCKOUT_SECONDS = 300
MAX_OTP_ATTEMPTS = 3
MAX_OTP_REQUESTS = 10
OTP_REQUEST_WINDOW_SECONDS = 300
MAX_SECURITY_ANSWER_ATTEMPTS = 3
SECURITY_LOCKOUT_SECONDS = 300

COLORS = {
    "bg": "#0B0F19",
    "surface": "#111827",
    "surface_glass": "rgba(17, 24, 39, 0.75)",
    "sidebar": "#090D16",
    "card": "rgba(22, 30, 49, 0.7)",
    "card_border": "rgba(255, 255, 255, 0.08)",
    "ink": "#F8FAFC",
    "text": "#94A3B8",
    "text_light": "#E2E8F0",
    "muted": "#64748B",
    "primary": "#4F46E5",
    "primary_gradient": "linear-gradient(135deg, #4F46E5 0%, #7C3AED 100%)",
    "primary_hover": "linear-gradient(135deg, #4338CA 0%, #6D28D9 100%)",
    "accent_cyan": "#06B6D4",
    "accent_emerald": "#10B981",
    "accent_rose": "#F43F5E",
}


def apply_theme():
    st.set_page_config(
        page_title=APP_NAME,
        page_icon="🛡️",
        layout="wide",
        initial_sidebar_state="expanded"
    )

    st.markdown(
        f"""
        <style>
        @import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700;800&family=Inter:wght@300;400;500;600;700&display=swap');

        /* Global App Styling */
        html, body, .stApp {{
            background: {COLORS['bg']};
            background-image: 
                radial-gradient(circle at 15% 15%, rgba(79, 70, 229, 0.12) 0%, transparent 40%),
                radial-gradient(circle at 85% 25%, rgba(6, 182, 212, 0.08) 0%, transparent 45%),
                radial-gradient(circle at 50% 85%, rgba(124, 58, 237, 0.1) 0%, transparent 50%);
            background-attachment: fixed;
            color: {COLORS['text_light']};
            font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif;
            letter-spacing: -0.01em;
        }}

        /* Typography */
        h1, h2, h3, h4, h5, h6 {{
            font-family: 'Plus Jakarta Sans', sans-serif !important;
            font-weight: 700;
            color: #FFFFFF !important;
            letter-spacing: -0.02em;
        }}

        p, span, label, div {{
            font-family: 'Inter', sans-serif;
        }}

        .block-container {{
            max-width: 1200px;
            padding-top: 2rem;
            padding-bottom: 3.5rem;
        }}

        /* Top Bar & Header Cleanup */
        header[data-testid="stHeader"] {{
            background: transparent !important;
        }}

        /* Sidebar Styling */
        section[data-testid="stSidebar"] {{
            background: {COLORS['sidebar']} !important;
            border-right: 1px solid rgba(255, 255, 255, 0.06);
            backdrop-filter: blur(20px);
        }}
        
        section[data-testid="stSidebar"] div.block-container {{
            padding-top: 2rem;
            padding-left: 1.5rem;
            padding-right: 1.5rem;
        }}

        /* Modern Radio Buttons for Navigation */
        div[data-testid="stRadio"] > div {{
            background: rgba(255, 255, 255, 0.03);
            border: 1px solid rgba(255, 255, 255, 0.06);
            border-radius: 12px;
            padding: 6px;
            gap: 4px;
        }}

        div[data-testid="stRadio"] label {{
            background: transparent !important;
            border-radius: 8px !important;
            padding: 8px 14px !important;
            margin: 2px 0 !important;
            color: #94A3B8 !important;
            font-weight: 500 !important;
            transition: all 0.2s ease !important;
            cursor: pointer !important;
        }}

        div[data-testid="stRadio"] label:hover {{
            background: rgba(255, 255, 255, 0.06) !important;
            color: #FFFFFF !important;
        }}

        div[data-testid="stRadio"] label[data-checked="true"],
        div[data-testid="stRadio"] label:has(input:checked) {{
            background: rgba(79, 70, 229, 0.25) !important;
            border: 1px solid rgba(99, 102, 241, 0.4) !important;
            color: #FFFFFF !important;
            font-weight: 600 !important;
        }}

        /* Custom Form Inputs */
        div[data-baseweb="input"],
        div[data-baseweb="select"] > div {{
            background: rgba(15, 23, 42, 0.75) !important;
            border: 1px solid rgba(255, 255, 255, 0.12) !important;
            border-radius: 12px !important;
            color: #FFFFFF !important;
            transition: all 0.25s cubic-bezier(0.4, 0, 0.2, 1) !important;
        }}

        div[data-baseweb="input"]:focus-within,
        div[data-baseweb="select"]:focus-within > div {{
            border-color: #6366F1 !important;
            box-shadow: 0 0 0 3px rgba(99, 102, 241, 0.22), 0 4px 12px rgba(79, 70, 229, 0.15) !important;
            background: rgba(15, 23, 42, 0.95) !important;
        }}

        input {{
            color: #FFFFFF !important;
            font-size: 0.95rem !important;
        }}

        input::placeholder {{
            color: #64748B !important;
        }}

        label[data-testid="stWidgetLabel"] p {{
            color: #CBD5E1 !important;
            font-weight: 500 !important;
            font-size: 0.88rem !important;
            margin-bottom: 4px;
        }}

        /* Form Container Polish */
        div[data-testid="stForm"] {{
            background: {COLORS['surface_glass']} !important;
            border: 1px solid {COLORS['card_border']} !important;
            border-radius: 20px !important;
            padding: 28px !important;
            box-shadow: 0 20px 40px -15px rgba(0, 0, 0, 0.6), inset 0 1px 0 rgba(255, 255, 255, 0.08) !important;
            backdrop-filter: blur(16px) !important;
        }}

        /* Buttons Styling */
        div[data-testid="stButton"] button,
        div[data-testid="stFormSubmitButton"] button {{
            background: {COLORS['primary_gradient']} !important;
            color: #FFFFFF !important;
            border: 1px solid rgba(255, 255, 255, 0.15) !important;
            border-radius: 12px !important;
            font-family: 'Plus Jakarta Sans', sans-serif !important;
            font-weight: 600 !important;
            font-size: 0.95rem !important;
            letter-spacing: 0.01em !important;
            min-height: 46px !important;
            box-shadow: 0 4px 14px rgba(79, 70, 229, 0.35) !important;
            transition: all 0.25s cubic-bezier(0.4, 0, 0.2, 1) !important;
        }}

        div[data-testid="stButton"] button:hover,
        div[data-testid="stFormSubmitButton"] button:hover {{
            background: {COLORS['primary_hover']} !important;
            transform: translateY(-1.5px) !important;
            box-shadow: 0 8px 24px rgba(79, 70, 229, 0.5) !important;
            border-color: rgba(255, 255, 255, 0.3) !important;
        }}

        div[data-testid="stButton"] button:active,
        div[data-testid="stFormSubmitButton"] button:active {{
            transform: translateY(0.5px) !important;
            box-shadow: 0 2px 8px rgba(79, 70, 229, 0.4) !important;
        }}

        /* Secondary / Outline Button */
        div[data-testid="stButton"] button[kind="secondary"] {{
            background: rgba(30, 41, 59, 0.6) !important;
            color: #E2E8F0 !important;
            border: 1px solid rgba(255, 255, 255, 0.1) !important;
            box-shadow: none !important;
        }}

        div[data-testid="stButton"] button[kind="secondary"]:hover {{
            background: rgba(51, 65, 85, 0.8) !important;
            color: #FFFFFF !important;
            border-color: rgba(255, 255, 255, 0.2) !important;
            transform: translateY(-1px) !important;
        }}

        /* Metrics Cards */
        div[data-testid="stMetric"] {{
            background: rgba(17, 24, 39, 0.65) !important;
            border: 1px solid rgba(255, 255, 255, 0.07) !important;
            border-radius: 16px !important;
            padding: 18px 22px !important;
            box-shadow: 0 10px 25px -5px rgba(0, 0, 0, 0.4) !important;
            backdrop-filter: blur(12px) !important;
            transition: transform 0.2s ease, border-color 0.2s ease;
        }}

        div[data-testid="stMetric"]:hover {{
            transform: translateY(-2px);
            border-color: rgba(99, 102, 241, 0.3) !important;
        }}

        div[data-testid="stMetricLabel"] p {{
            color: #94A3B8 !important;
            font-size: 0.85rem !important;
            font-weight: 500 !important;
            text-transform: uppercase;
            letter-spacing: 0.05em;
        }}

        div[data-testid="stMetricValue"] {{
            color: #FFFFFF !important;
            font-family: 'Plus Jakarta Sans', sans-serif !important;
            font-weight: 700 !important;
            font-size: 1.85rem !important;
        }}

        /* Alert / Messages Polish */
        div[data-testid="stAlert"] {{
            border-radius: 14px !important;
            backdrop-filter: blur(12px) !important;
            border-width: 1px !important;
            font-weight: 500 !important;
        }}

        /* Dataframe Modern Styling */
        div[data-testid="stDataFrame"] {{
            background: rgba(17, 24, 39, 0.7) !important;
            border: 1px solid rgba(255, 255, 255, 0.08) !important;
            border-radius: 16px !important;
            overflow: hidden !important;
            box-shadow: 0 12px 30px rgba(0, 0, 0, 0.4) !important;
        }}

        /* Glass Cards Custom Classes */
        .glass-card {{
            background: rgba(17, 24, 39, 0.75);
            border: 1px solid rgba(255, 255, 255, 0.08);
            border-radius: 20px;
            padding: 26px;
            box-shadow: 0 20px 40px -15px rgba(0, 0, 0, 0.6), inset 0 1px 0 rgba(255, 255, 255, 0.08);
            backdrop-filter: blur(16px);
        }}

        .brand-badge {{
            display: inline-flex;
            align-items: center;
            gap: 8px;
            background: rgba(79, 70, 229, 0.15);
            border: 1px solid rgba(99, 102, 241, 0.35);
            color: #818CF8;
            font-size: 0.75rem;
            font-weight: 700;
            text-transform: uppercase;
            letter-spacing: 0.08em;
            padding: 5px 14px;
            border-radius: 9999px;
            margin-bottom: 12px;
        }}

        .pulse-dot {{
            width: 7px;
            height: 7px;
            border-radius: 50%;
            background: #10B981;
            box-shadow: 0 0 10px #10B981;
            animation: pulse 2s infinite;
        }}

        @keyframes pulse {{
            0% {{ transform: scale(0.95); box-shadow: 0 0 0 0 rgba(16, 185, 129, 0.7); }}
            70% {{ transform: scale(1); box-shadow: 0 0 0 8px rgba(16, 185, 129, 0); }}
            100% {{ transform: scale(0.95); box-shadow: 0 0 0 0 rgba(16, 185, 129, 0); }}
        }}

        .gradient-title {{
            background: linear-gradient(135deg, #FFFFFF 0%, #CBD5E1 60%, #818CF8 100%);
            -webkit-background-clip: text;
            -webkit-text-fill-color: transparent;
            font-size: 2.2rem;
            font-weight: 800;
            margin: 0;
            line-height: 1.2;
        }}

        .gradient-subtext {{
            color: #94A3B8;
            font-size: 0.95rem;
            margin-top: 6px;
        }}

        .otp-display-box {{
            background: linear-gradient(135deg, rgba(79, 70, 229, 0.2) 0%, rgba(6, 182, 212, 0.15) 100%);
            border: 1.5px dashed rgba(99, 102, 241, 0.5);
            border-radius: 14px;
            padding: 16px;
            text-align: center;
            margin: 14px 0;
        }}

        .feature-chip {{
            display: inline-flex;
            align-items: center;
            gap: 6px;
            padding: 4px 10px;
            border-radius: 8px;
            background: rgba(255, 255, 255, 0.05);
            border: 1px solid rgba(255, 255, 255, 0.08);
            font-size: 0.78rem;
            color: #CBD5E1;
        }}

        /* -------------------------------------------
           Exact White OTP Verification Card Styling 
           ------------------------------------------- */
        .white-verification-card {{
            background: #FFFFFF !important;
            border-radius: 20px !important;
            padding: 36px 42px !important;
            box-shadow: 0 25px 60px -12px rgba(15, 23, 42, 0.45) !important;
            color: #1E293B !important;
            margin: 10px auto 20px !important;
        }}

        .white-verification-card h2 {{
            color: #1E293B !important;
            font-family: 'Plus Jakarta Sans', sans-serif !important;
            font-weight: 700 !important;
        }}

        .white-verification-card p,
        .white-verification-card span,
        .white-verification-card div {{
            color: #64748B;
        }}

        /* 6-Digit OTP Boxes Container */
        .otp-boxes-grid div[data-baseweb="input"] {{
            border: 2px solid #10B981 !important;
            border-radius: 10px !important;
            background: #FFFFFF !important;
            height: 56px !important;
            box-shadow: 0 2px 6px rgba(16, 185, 129, 0.12) !important;
            transition: all 0.2s cubic-bezier(0.4, 0, 0.2, 1) !important;
        }}

        .otp-boxes-grid div[data-baseweb="input"]:focus-within {{
            border-color: #059669 !important;
            box-shadow: 0 0 0 3px rgba(16, 185, 129, 0.28) !important;
            transform: translateY(-1px) !important;
        }}

        .otp-boxes-grid input {{
            text-align: center !important;
            font-size: 26px !important;
            font-weight: 700 !important;
            color: #1E293B !important;
            padding: 0 !important;
            font-family: 'Plus Jakarta Sans', -apple-system, sans-serif !important;
        }}
        </style>
        """,
        unsafe_allow_html=True
    )


In [ ]:
# @title Database
%%writefile database.py
import sqlite3
import datetime as dt
import time
import os
from config import DB_PATH, ADMIN_USERNAME, ADMIN_EMAIL, ADMIN_PASSWORD, MAX_SECURITY_ANSWER_ATTEMPTS, SECURITY_LOCKOUT_SECONDS
import bcrypt


def db_connect():
    conn = sqlite3.connect(
        DB_PATH, timeout=15, check_same_thread=False
    )
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


def utc_now():
    return dt.datetime.now(
        dt.timezone.utc
    ).isoformat(timespec="seconds")


def hash_password(value):
    return bcrypt.hashpw(
        value.encode("utf-8"),
        bcrypt.gensalt()
    ).decode("utf-8")


def check_password(value, hashed):
    if not value or not hashed:
        return False

    try:
        return bcrypt.checkpw(
            value.encode("utf-8"),
            hashed.encode("utf-8")
        )
    except (ValueError, TypeError):
        return False


def init_db():
    with db_connect() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS users (
                email TEXT PRIMARY KEY COLLATE NOCASE,
                username TEXT NOT NULL COLLATE NOCASE,
                password_hash TEXT NOT NULL,
                created_at TEXT NOT NULL,
                role TEXT NOT NULL DEFAULT 'user',
                security_question TEXT,
                security_answer_hash TEXT
            )
        """)

        conn.execute("""
            CREATE TABLE IF NOT EXISTS password_history (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                email TEXT NOT NULL COLLATE NOCASE,
                password_hash TEXT NOT NULL,
                set_at TEXT NOT NULL,
                FOREIGN KEY(email)
                    REFERENCES users(email)
                    ON DELETE CASCADE
            )
        """)

        conn.execute("""
            CREATE TABLE IF NOT EXISTS login_attempts (
                email TEXT PRIMARY KEY COLLATE NOCASE,
                attempts INTEGER NOT NULL DEFAULT 0,
                last_attempt REAL NOT NULL DEFAULT 0
            )
        """)

        conn.execute("""
            CREATE TABLE IF NOT EXISTS otp_requests (
                email TEXT NOT NULL COLLATE NOCASE,
                requested_at REAL NOT NULL
            )
        """)

        conn.execute("""
            CREATE TABLE IF NOT EXISTS otp_attempts (
                email TEXT PRIMARY KEY COLLATE NOCASE,
                attempts INTEGER NOT NULL DEFAULT 0,
                last_attempt REAL NOT NULL DEFAULT 0
            )
        """)

        conn.execute("""
            CREATE TABLE IF NOT EXISTS security_attempts (
                email TEXT PRIMARY KEY COLLATE NOCASE,
                attempts INTEGER NOT NULL DEFAULT 0,
                last_attempt REAL NOT NULL DEFAULT 0
            )
        """)

        # Ensure default Admin account defined in code exists (Step 11)
        admin_email = (ADMIN_EMAIL or "admin@portal.local").strip().lower()
        admin_uname = (ADMIN_USERNAME or "admin").strip()
        admin_pwd = ADMIN_PASSWORD or "Admin@123"

        existing_admin = conn.execute(
            "SELECT 1 FROM users WHERE email = ? OR username = ? COLLATE NOCASE",
            (admin_email, admin_uname)
        ).fetchone()

        if not existing_admin:
            admin_hash = hash_password(admin_pwd)
            now = utc_now()

            conn.execute("""
                INSERT INTO users
                (email, username, password_hash, created_at, role)
                VALUES (?, ?, ?, ?, 'admin')
            """, (
                admin_email,
                admin_uname,
                admin_hash,
                now
            ))

            conn.execute("""
                INSERT INTO password_history
                (email, password_hash, set_at)
                VALUES (?, ?, ?)
            """, (admin_email, admin_hash, now))
        else:
            # Ensure admin username is synced to admin_uname (e.g. 'admin')
            conn.execute("""
                UPDATE users
                SET username = ?
                WHERE email = ? AND role = 'admin'
            """, (admin_uname, admin_email))


def get_user(identifier):
    if not identifier:
        return None
    val = str(identifier).strip()
    with db_connect() as conn:
        return conn.execute(
            "SELECT * FROM users WHERE email = ? COLLATE NOCASE OR username = ? COLLATE NOCASE",
            (val, val)
        ).fetchone()


def get_user_by_username(username):
    if not username:
        return None
    with db_connect() as conn:
        return conn.execute(
            "SELECT * FROM users WHERE username = ? COLLATE NOCASE",
            (str(username).strip(),)
        ).fetchone()


def get_all_users_for_admin():
    """Returns list of registered users for Admin Dashboard without password data."""
    with db_connect() as conn:
        return conn.execute(
            "SELECT username, email, role, created_at FROM users ORDER BY created_at DESC"
        ).fetchall()


def register_user(
    username, email, password, question, answer
):
    username = username.strip()
    email = email.lower().strip()

    if get_user_by_username(username):
        return False, "This username is already taken. Please choose another."

    now = utc_now()
    hashed = hash_password(password)

    answer_hash = (
        hash_password(answer.strip().lower())
        if answer.strip()
        else None
    )

    try:
        with db_connect() as conn:
            conn.execute("""
                INSERT INTO users
                (
                    email, username, password_hash,
                    created_at, role,
                    security_question, security_answer_hash
                )
                VALUES (?, ?, ?, ?, 'user', ?, ?)
            """, (
                email,
                username.strip(),
                hashed,
                now,
                question,
                answer_hash
            ))

            conn.execute("""
                INSERT INTO password_history
                (email, password_hash, set_at)
                VALUES (?, ?, ?)
            """, (email, hashed, now))

        return True, "Account created."

    except sqlite3.IntegrityError:
        return False, (
            "An account with that email already exists."
        )


def update_password(email, new_password):
    hashed = hash_password(new_password)
    now = utc_now()
    email = email.lower().strip()

    with db_connect() as conn:
        conn.execute("""
            UPDATE users
            SET password_hash = ?
            WHERE email = ?
        """, (hashed, email))

        conn.execute("""
            INSERT INTO password_history
            (email, password_hash, set_at)
            VALUES (?, ?, ?)
        """, (email, hashed, now))

        conn.execute(
            "DELETE FROM login_attempts WHERE email = ?",
            (email,)
        )


def get_otp_attempts(email):
    email = str(email or "").lower().strip()
    with db_connect() as conn:
        row = conn.execute(
            "SELECT attempts FROM otp_attempts WHERE email = ?",
            (email,)
        ).fetchone()
        return row["attempts"] if row else 0


def record_failed_otp_attempt(email):
    email = str(email or "").lower().strip()
    with db_connect() as conn:
        row = conn.execute(
            "SELECT attempts FROM otp_attempts WHERE email = ?",
            (email,)
        ).fetchone()
        attempts = (row["attempts"] if row else 0) + 1
        now = time.time()
        conn.execute("""
            INSERT INTO otp_attempts (email, attempts, last_attempt)
            VALUES (?, ?, ?)
            ON CONFLICT(email) DO UPDATE SET
                attempts = excluded.attempts,
                last_attempt = excluded.last_attempt
        """, (email, attempts, now))
        return attempts


def clear_otp_attempts(email):
    email = str(email or "").lower().strip()
    with db_connect() as conn:
        conn.execute("DELETE FROM otp_attempts WHERE email = ?", (email,))


def get_security_lockout_remaining(email):
    email = str(email or "").lower().strip()
    with db_connect() as conn:
        row = conn.execute(
            "SELECT attempts, last_attempt FROM security_attempts WHERE email = ?",
            (email,)
        ).fetchone()
    if not row or row["attempts"] < MAX_SECURITY_ANSWER_ATTEMPTS:
        return 0
    remaining = SECURITY_LOCKOUT_SECONDS - (time.time() - row["last_attempt"])
    if remaining <= 0:
        with db_connect() as conn:
            conn.execute("DELETE FROM security_attempts WHERE email = ?", (email,))
        return 0
    return int(remaining) + 1


def record_failed_security_attempt(email):
    email = str(email or "").lower().strip()
    with db_connect() as conn:
        row = conn.execute(
            "SELECT attempts FROM security_attempts WHERE email = ?",
            (email,)
        ).fetchone()
        attempts = (row["attempts"] if row else 0) + 1
        now = time.time()
        conn.execute("""
            INSERT INTO security_attempts (email, attempts, last_attempt)
            VALUES (?, ?, ?)
            ON CONFLICT(email) DO UPDATE SET
                attempts = excluded.attempts,
                last_attempt = excluded.last_attempt
        """, (email, attempts, now))
        return attempts


def clear_security_attempts(email):
    email = str(email or "").lower().strip()
    with db_connect() as conn:
        conn.execute("DELETE FROM security_attempts WHERE email = ?", (email,))

In [ ]:
# @title Authentication
%%writefile auth.py
import time
import datetime as dt
import jwt

from config import (
    JWT_SECRET,
    MAX_LOGIN_ATTEMPTS,
    LOCKOUT_SECONDS
)
from database import db_connect


def login_lockout_remaining(email):
    email = email.lower().strip()

    with db_connect() as conn:
        row = conn.execute("""
            SELECT attempts, last_attempt
            FROM login_attempts
            WHERE email = ?
        """, (email,)).fetchone()

    if (
        not row
        or row["attempts"] < MAX_LOGIN_ATTEMPTS
    ):
        return 0

    remaining = LOCKOUT_SECONDS - (
        time.time() - row["last_attempt"]
    )

    if remaining <= 0:
        with db_connect() as conn:
            conn.execute(
                "DELETE FROM login_attempts WHERE email = ?",
                (email,)
            )
        return 0

    return int(remaining) + 1


def record_failed_login(email):
    email = email.lower().strip()

    with db_connect() as conn:
        row = conn.execute("""
            SELECT attempts
            FROM login_attempts
            WHERE email = ?
        """, (email,)).fetchone()

        attempts = (
            row["attempts"] if row else 0
        ) + 1

        conn.execute("""
            INSERT INTO login_attempts
            (email, attempts, last_attempt)
            VALUES (?, ?, ?)
            ON CONFLICT(email) DO UPDATE SET
                attempts = excluded.attempts,
                last_attempt = excluded.last_attempt
        """, (email, attempts, time.time()))


def clear_failed_logins(email):
    with db_connect() as conn:
        conn.execute(
            "DELETE FROM login_attempts WHERE email = ?",
            (email.lower().strip(),)
        )


def issue_login_token(email):
    if not JWT_SECRET:
        raise RuntimeError(
            "JWT_SECRET is not configured."
        )

    now = dt.datetime.now(dt.timezone.utc)

    return jwt.encode(
        {
            "sub": email.lower().strip(),
            "iat": now,
            "exp": now + dt.timedelta(hours=2),
            "type": "login"
        },
        JWT_SECRET,
        algorithm="HS256"
    )


def decode_login_token(token):
    if not token or not JWT_SECRET:
        return None

    try:
        payload = jwt.decode(
            token,
            JWT_SECRET,
            algorithms=["HS256"]
        )

        if payload.get("type") == "login":
            return payload

        return None

    except jwt.PyJWTError:
        return None

In [ ]:
# @title OTP
%%writefile otp.py
import time
import secrets
import datetime as dt
import smtplib
import jwt

from email.utils import formatdate, make_msgid
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

from config import (
    JWT_SECRET,
    SENDER_EMAIL,
    EMAIL_PASSWORD,
    OTP_EXPIRY_MINUTES,
    MAX_OTP_ATTEMPTS,
    MAX_OTP_REQUESTS,
    OTP_REQUEST_WINDOW_SECONDS
)

from database import (
    db_connect,
    hash_password,
    check_password,
    get_otp_attempts,
    record_failed_otp_attempt,
    clear_otp_attempts
)


def generate_otp():
    return str(
        secrets.randbelow(900000) + 100000
    )


def make_otp_token(email, otp):
    if not JWT_SECRET:
        raise RuntimeError(
            "JWT_SECRET is not configured."
        )

    now = dt.datetime.now(dt.timezone.utc)

    payload = {
        "sub": email.lower().strip(),
        "otp_hash": hash_password(otp),
        "type": "password_reset_otp",
        "iat": now,
        "exp": now + dt.timedelta(
            minutes=OTP_EXPIRY_MINUTES
        )
    }

    return jwt.encode(
        payload,
        JWT_SECRET,
        algorithm="HS256"
    )


def verify_otp_token(token, input_otp, email):
    email = str(email or "").lower().strip()
    if not token:
        return False, "This verification session has expired. Please request a new code."

    current_attempts = get_otp_attempts(email)
    if current_attempts >= MAX_OTP_ATTEMPTS:
        return False, "Too many incorrect attempts. This code has been invalidated. Please request a new code."

    clean_otp = str(input_otp or "").strip()
    if not clean_otp or len(clean_otp) != 6 or not clean_otp.isdigit():
        return False, "Please enter the complete 6-digit verification code."

    try:
        payload = jwt.decode(
            token,
            JWT_SECRET,
            algorithms=["HS256"]
        )

        if (
            payload.get("sub") != email
            or payload.get("type") != "password_reset_otp"
        ):
            return False, (
                "Verification token mismatch. "
                "Request a new code."
            )

        if check_password(
            clean_otp,
            payload.get("otp_hash")
        ):
            clear_otp_attempts(email)
            return True, "Verified."

        new_attempts = record_failed_otp_attempt(email)
        remaining = MAX_OTP_ATTEMPTS - new_attempts
        if remaining > 0:
            return False, f"Incorrect code. You have {remaining} attempt{'s' if remaining > 1 else ''} remaining."
        else:
            return False, "Too many incorrect attempts. This code has been invalidated. Please request a new code."

    except jwt.ExpiredSignatureError:
        return False, (
            f"The code expired after "
            f"{OTP_EXPIRY_MINUTES} minutes. "
            "Request a new code."
        )

    except jwt.PyJWTError:
        return False, (
            "Verification session is invalid. "
            "Request a new code."
        )


def can_request_otp(email):
    cutoff = (
        time.time() - OTP_REQUEST_WINDOW_SECONDS
    )

    with db_connect() as conn:
        conn.execute(
            "DELETE FROM otp_requests WHERE requested_at < ?",
            (cutoff,)
        )

        row = conn.execute("""
            SELECT COUNT(*) AS n
            FROM otp_requests
            WHERE email = ?
              AND requested_at >= ?
        """, (
            email.lower().strip(),
            cutoff
        )).fetchone()

    return row["n"] < MAX_OTP_REQUESTS


def record_otp_request(email):
    with db_connect() as conn:
        conn.execute("""
            INSERT INTO otp_requests
            (email, requested_at)
            VALUES (?, ?)
        """, (
            email.lower().strip(),
            time.time()
        ))


def send_otp_email(to_email, otp, purpose="reset"):
    if not SENDER_EMAIL or not EMAIL_PASSWORD:
        return False, (
            "Email settings are missing. Add EMAIL_ID "
            "and EMAIL_PASSWORD to Colab Secrets."
        )

    is_signup = str(purpose).lower() in ("signup", "create", "register", "account")

    if is_signup:
        badge_text = "Account Verification"
        heading = "Verify Your Account"
        subject = "Your Verify Portal verification code"
        intro = (
            "Thank you for registering! Please use the 6-digit verification code below "
            "to complete your account setup and activate your portal access."
        )
        security_note = "If you didn't attempt to create an account, you can safely ignore this email."
        icon_symbol = "✦"
    else:
        badge_text = "Password Reset"
        heading = "Reset Your Password"
        subject = "Your Verify Portal password reset code"
        intro = (
            "We received a request to reset the password for your account. "
            "Please use the 6-digit verification code below to authorize this change."
        )
        security_note = "If you didn't request a password reset, you can safely ignore this email. Your password will remain unchanged."
        icon_symbol = "🔒"

    msg = MIMEMultipart("alternative")
    msg["From"] = f"Verify Portal <{SENDER_EMAIL}>"
    msg["To"] = to_email
    msg["Subject"] = subject
    msg["Date"] = formatdate(localtime=True)
    msg["Message-ID"] = make_msgid()
    msg["Reply-To"] = SENDER_EMAIL

    text_body = (
        f"Hi,\n\n"
        f"Your verification code is:\n\n"
        f"{otp}\n\n"
        f"This code expires in {OTP_EXPIRY_MINUTES} minutes.\n\n"
        f"Don't reply to this email or share this code with anyone. Verify Portal will never ask you for it.\n\n"
        f"{security_note}\n\n"
        f"--\nVerify Portal\n"
    )

    html_body = f"""<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>{subject}</title>
</head>
<body style="margin: 0; padding: 0; background-color: #f1f3f9; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; -webkit-font-smoothing: antialiased;">
  <table role="presentation" width="100%" border="0" cellspacing="0" cellpadding="0" style="background-color: #f1f3f9; padding: 36px 12px;">
    <tr>
      <td align="center">
        <!-- Main Card Container -->
        <table role="presentation" width="100%" border="0" cellspacing="0" cellpadding="0" style="max-width: 540px; background-color: #ffffff; border-radius: 16px; border: 1px solid #e2e8f0; box-shadow: 0 12px 36px rgba(98, 96, 230, 0.08); overflow: hidden;">
          
          <!-- Top Accent Banner -->
          <tr>
            <td style="height: 6px; background: linear-gradient(90deg, #6260e6, #8343ad);"></td>
          </tr>

          <!-- Main Content -->
          <tr>
            <td style="padding: 42px 38px 36px; text-align: center;">

              <!-- Glowing Icon Circle -->
              <div style="display: inline-block; width: 64px; height: 64px; line-height: 64px; border-radius: 50%; background: linear-gradient(135deg, #6260e6 0%, #8343ad 100%); box-shadow: 0 8px 24px rgba(98, 96, 230, 0.35); margin-bottom: 20px; text-align: center;">
                <span style="font-size: 28px; color: #ffffff; vertical-align: middle;">{icon_symbol}</span>
              </div>

              <!-- Brand Name -->
              <div style="font-size: 13px; font-weight: 800; color: #6260e6; letter-spacing: 2px; text-transform: uppercase; margin-bottom: 10px;">
                VERIFY PORTAL
              </div>

              <!-- Sleek Badge -->
              <div style="margin-bottom: 16px;">
                <span style="display: inline-block; background-color: #ede9fe; color: #5b21b6; border: 1px solid #ddd6fe; border-radius: 20px; padding: 4px 15px; font-size: 11px; font-weight: 700; letter-spacing: 0.8px; text-transform: uppercase;">
                  {badge_text}
                </span>
              </div>

              <!-- Main Heading -->
              <h1 style="margin: 0 0 12px; font-size: 25px; font-weight: 800; color: #1e1b4b; letter-spacing: -0.5px; line-height: 1.3;">
                {heading}
              </h1>

              <!-- Intro Subtitle -->
              <p style="margin: 0 auto 20px; font-size: 14.5px; color: #64748b; line-height: 1.6; max-width: 440px;">
                {intro}
              </p>

              <!-- Verification Code Container (Formatted for Gmail NLP Extraction) -->
              <div style="background: linear-gradient(180deg, #f8fafc 0%, #f1f5f9 100%); border: 2px dashed #6260e6; border-radius: 14px; padding: 24px 20px; margin: 24px 0; text-align: center;">
                <p style="margin: 0 0 12px; font-size: 14px; font-weight: 700; color: #1e1b4b; text-transform: uppercase; letter-spacing: 1px;">
                  Your verification code is:
                </p>

                <!-- Isolated Digits Box -->
                <div style="display: inline-block; background-color: #ffffff; border: 1.5px solid #e0e7ff; border-radius: 10px; padding: 14px 32px; box-shadow: 0 4px 14px rgba(98, 96, 230, 0.08); margin-bottom: 12px; user-select: all; -webkit-user-select: all; -moz-user-select: all;">
                  <span id="otpCode" style="font-family: 'SF Mono', Consolas, 'Liberation Mono', Menlo, Courier, monospace; font-size: 38px; font-weight: 800; letter-spacing: 8px; color: #1e1b4b; user-select: all; -webkit-user-select: all; -moz-user-select: all;">{otp}</span>
                </div>

                <div style="font-size: 13px; color: #64748b; margin-top: 4px;">
                  This code expires in <strong>{OTP_EXPIRY_MINUTES} minutes</strong>.
                </div>
              </div>

              <!-- Expiry Alert -->
              <table role="presentation" width="100%" border="0" cellspacing="0" cellpadding="0" style="background-color: #fdf4ff; border: 1px solid #f0abfc; border-radius: 10px; margin-bottom: 24px;">
                <tr>
                  <td style="padding: 12px 16px; text-align: left;">
                    <span style="font-size: 16px; vertical-align: middle; margin-right: 8px;">⏱</span>
                    <span style="font-size: 13px; font-weight: 600; color: #86198f; vertical-align: middle;">
                      This code is valid for <strong>{OTP_EXPIRY_MINUTES} minutes</strong>. Please do not share it.
                    </span>
                  </td>
                </tr>
              </table>

              <!-- Divider -->
              <div style="height: 1px; background-color: #e2e8f0; margin: 24px 0;"></div>

              <!-- Security Information -->
              <div style="text-align: left; font-size: 12.5px; line-height: 1.6; color: #64748b;">
                <p style="margin: 0 0 8px;">
                  🔒 <strong>Security Notice:</strong> Verify staff will never ask for your verification code or password.
                </p>
                <p style="margin: 0; color: #94a3b8;">
                  {security_note}
                </p>
              </div>

            </td>
          </tr>

          <!-- Footer -->
          <tr>
            <td style="background-color: #f8fafc; border-top: 1px solid #edf2f7; padding: 20px 36px; text-align: center; font-size: 11.5px; color: #94a3b8; line-height: 1.5;">
              <strong style="color: #64748b;">Verify Authentication Services</strong><br>
              Cryptographic One-Time Passcode Verification • Automatic Notification<br>
              This is an automated security message. Please do not reply directly.
            </td>
          </tr>
        </table>
      </td>
    </tr>
  </table>
</body>
</html>
"""

    msg.attach(MIMEText(text_body, "plain"))
    msg.attach(MIMEText(html_body, "html"))

    try:
        with smtplib.SMTP(
            "smtp.gmail.com", 587, timeout=20
        ) as server:
            server.ehlo()
            server.starttls()
            server.ehlo()
            server.login(
                SENDER_EMAIL,
                EMAIL_PASSWORD
            )
            server.sendmail(
                SENDER_EMAIL,
                [to_email],
                msg.as_string()
            )

        clear_otp_attempts(to_email)
        return True, "Email sent."

    except Exception:
        return False, (
            "The email could not be sent. Check EMAIL_ID "
            "and the Gmail App Password in Colab Secrets."
        )

In [ ]:
# @title UI
%%writefile ui.py
import re
import textwrap
import time
from pathlib import Path
import streamlit as st
import streamlit.components.v1 as components

from config import (
    APP_NAME,
    JWT_SECRET,
    OTP_EXPIRY_MINUTES,
    MAX_SECURITY_ANSWER_ATTEMPTS,
    SECURITY_LOCKOUT_SECONDS,
    COLORS
)

from database import (
    get_user,
    get_user_by_username,
    get_all_users_for_admin,
    register_user,
    check_password,
    update_password,
    db_connect,
    get_security_lockout_remaining,
    record_failed_security_attempt,
    clear_security_attempts
)

from auth import (
    login_lockout_remaining,
    record_failed_login,
    clear_failed_logins,
    issue_login_token
)

from otp import (
    generate_otp,
    make_otp_token,
    verify_otp_token,
    send_otp_email,
    can_request_otp,
    record_otp_request
)


otp_verification_component = components.declare_component(
    "otp_verification",
    path=str(Path(__file__).parent / "otp_verification_component"),
)

reset_email_component = components.declare_component(
    "reset_email",
    path=str(Path(__file__).parent / "reset_email_component"),
)

reset_password_component = components.declare_component(
    "reset_password",
    path=str(Path(__file__).parent / "reset_password_component"),
)

security_recovery_component = components.declare_component(
    "security_recovery",
    path=str(Path(__file__).parent / "security_recovery_component"),
)

login_component = components.declare_component(
    "login",
    path=str(Path(__file__).parent / "login_component"),
)

signup_component = components.declare_component(
    "signup", path=str(Path(__file__).parent / "signup_component")
)

signup_otp_component = components.declare_component(
    "signup_otp", path=str(Path(__file__).parent / "signup_otp_component")
)

signup_complete_component = components.declare_component(
    "signup_complete", path=str(Path(__file__).parent / "signup_complete_component")
)


EMAIL_REGEX = r"^[a-zA-Z0-9._%+-]{2,}@[a-zA-Z0-9.-]{2,}\.[a-zA-Z]{2,}$"


def email_is_valid(email):
    """Enforces Step 8: at least 2 chars before @, 2 between @ and dot, and 2 after final dot."""
    return bool(
        re.fullmatch(
            EMAIL_REGEX,
            (email or "").strip()
        )
    )


def security_answer_is_valid(ans):
    """Validates security answers against gibberish, single spaced letters ('a v a a'), and short answers."""
    if not ans or not isinstance(ans, str):
        return False, "Security answer cannot be empty."
    clean = ans.strip()
    clean = re.sub(r"\s+", " ", clean)
    if len(clean) < 3:
        return False, "Security answer must be at least 3 characters long."
    alnum_only = re.sub(r"[^a-zA-Z0-9]", "", clean)
    if len(alnum_only) < 3:
        return False, "Security answer must contain at least 3 letters or numbers."
    words = clean.split()
    if len(words) >= 3 and all(len(w) == 1 for w in words):
        return False, "Security answer is invalid. Please enter a valid word or phrase (not spaced single letters)."
    if len(set(alnum_only.lower())) == 1 and len(alnum_only) > 2:
        return False, "Security answer cannot be a single repeating character."
    return True, ""


def check_password_strength(pwd):
    checks = {
        "length": len(pwd) >= 8,
        "uppercase": bool(re.search(r"[A-Z]", pwd)),
        "lowercase": bool(re.search(r"[a-z]", pwd)),
        "digit": bool(re.search(r"\d", pwd)),
        "special": bool(re.search(r"[!@#$%^&*(),.?\":{}|<>]", pwd)),
    }
    score = sum(checks.values())
    if score <= 2:
        level = "Weak"
        color = "#EF4444"
        percent = "30%"
    elif score <= 4:
        level = "Moderate"
        color = "#F59E0B"
        percent = "70%"
    else:
        level = "Strong"
        color = "#10B981"
        percent = "100%"
    return score, level, color, percent, checks


def render_password_meter(pwd):
    if not pwd:
        st.markdown(
            textwrap.dedent("""
            <div style="background: rgba(255, 255, 255, 0.03); border: 1px solid rgba(255, 255, 255, 0.07); border-radius: 12px; padding: 12px 14px; margin: 10px 0 16px;">
                <div style="font-size: 0.78rem; font-weight: 600; color: #94A3B8; text-transform: uppercase; letter-spacing: 0.05em; margin-bottom: 8px;">
                    Password Security Criteria:
                </div>
                <div style="display: flex; gap: 8px; flex-wrap: wrap;">
                    <span class="feature-chip">○ 8+ characters</span>
                    <span class="feature-chip">○ Upper & Lowercase</span>
                    <span class="feature-chip">○ Number (0-9)</span>
                    <span class="feature-chip">○ Symbol (!@#$)</span>
                </div>
            </div>
            """),
            unsafe_allow_html=True
        )
        return

    score, level, color, percent, checks = check_password_strength(pwd)
    
    chip_len = f'<span class="feature-chip" style="color:{"#10B981" if checks["length"] else "#94A3B8"}; border-color:{"rgba(16,185,129,0.3)" if checks["length"] else "rgba(255,255,255,0.08)"};">{"✓" if checks["length"] else "○"} 8+ chars</span>'
    chip_case = f'<span class="feature-chip" style="color:{"#10B981" if (checks["uppercase"] and checks["lowercase"]) else "#94A3B8"}; border-color:{"rgba(16,185,129,0.3)" if (checks["uppercase"] and checks["lowercase"]) else "rgba(255,255,255,0.08)"};">{"✓" if (checks["uppercase"] and checks["lowercase"]) else "○"} A-z Mix</span>'
    chip_num = f'<span class="feature-chip" style="color:{"#10B981" if checks["digit"] else "#94A3B8"}; border-color:{"rgba(16,185,129,0.3)" if checks["digit"] else "rgba(255,255,255,0.08)"};">{"✓" if checks["digit"] else "○"} Number</span>'
    chip_sym = f'<span class="feature-chip" style="color:{"#10B981" if checks["special"] else "#94A3B8"}; border-color:{"rgba(16,185,129,0.3)" if checks["special"] else "rgba(255,255,255,0.08)"};">{"✓" if checks["special"] else "○"} Symbol</span>'

    st.markdown(
        textwrap.dedent(f"""
        <div style="background: rgba(15, 23, 42, 0.7); border: 1px solid rgba(255, 255, 255, 0.08); border-radius: 12px; padding: 12px 14px; margin: 10px 0 16px;">
            <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 6px;">
                <span style="font-size: 0.8rem; font-weight: 600; color: #94A3B8;">Security Strength:</span>
                <span style="font-size: 0.82rem; font-weight: 700; color: {color};">{level}</span>
            </div>
            <div style="width: 100%; height: 6px; background: rgba(255, 255, 255, 0.1); border-radius: 999px; overflow: hidden; margin-bottom: 10px;">
                <div style="width: {percent}; height: 100%; background: {color}; transition: all 0.3s ease;"></div>
            </div>
            <div style="display: flex; gap: 6px; flex-wrap: wrap;">
                {chip_len} {chip_case} {chip_num} {chip_sym}
            </div>
        </div>
        """),
        unsafe_allow_html=True
    )


def render_step_indicator(current_stage):
    # Use native Streamlit elements here. Some Streamlit versions render the
    # inline HTML stepper as literal code, even with unsafe HTML enabled.
    steps = [("email", "Email"), ("otp", "Verify Code"), ("password", "New Password")]
    current_index = {"email": 1, "otp": 2, "password": 3}.get(current_stage, 1)
    st.progress(current_index / len(steps), text=f"Step {current_index} of {len(steps)}")
    columns = st.columns(len(steps))
    for index, (_, label) in enumerate(steps, start=1):
        with columns[index - 1]:
            marker = "✓" if index < current_index else str(index)
            prefix = "**" if index == current_index else ""
            suffix = "**" if index == current_index else ""
            st.caption(f"{prefix}{marker} {label}{suffix}")
    st.markdown("<div style='height: 10px;'></div>", unsafe_allow_html=True)
    return

    steps = [
        ("email", "1", "Email"),
        ("otp", "2", "Verify Code"),
        ("password", "3", "New Password"),
    ]
    stage_order = {"email": 1, "otp": 2, "password": 3}
    curr_num = stage_order.get(current_stage, 1)

    step_items = []
    for stage_key, num, title in steps:
        order = stage_order[stage_key]
        if order < curr_num:
            icon = "✓"
            bg = "linear-gradient(135deg, #10B981, #059669)"
            color = "#FFFFFF"
            border = "rgba(16, 185, 129, 0.4)"
            text_color = "#34D399"
        elif order == curr_num:
            icon = num
            bg = "linear-gradient(135deg, #4F46E5, #7C3AED)"
            color = "#FFFFFF"
            border = "rgba(124, 58, 237, 0.6)"
            text_color = "#FFFFFF"
        else:
            icon = num
            bg = "rgba(255, 255, 255, 0.05)"
            color = "#64748B"
            border = "rgba(255, 255, 255, 0.08)"
            text_color = "#64748B"

        step_items.append(
            f"""
            <div style="display: flex; align-items: center; gap: 7px;">
                <div style="
                    width: 26px; height: 26px; border-radius: 50%;
                    background: {bg}; color: {color};
                    display: flex; align-items: center; justify-content: center;
                    font-size: 0.78rem; font-weight: 700;
                    border: 1px solid {border};
                    box-shadow: { '0 0 10px rgba(99, 102, 241, 0.5)' if order == curr_num else 'none' };
                ">{icon}</div>
                <div style="font-size: 0.8rem; font-weight: 600; color: {text_color}; white-space: nowrap;">{title}</div>
            </div>
            """
        )

    divider = '<div style="flex: 1; height: 1.5px; background: rgba(255, 255, 255, 0.1); margin: 0 6px;"></div>'

    html = textwrap.dedent(f"""
    <div style="
        display: flex; align-items: center; justify-content: space-between;
        background: rgba(15, 23, 42, 0.6);
        border: 1px solid rgba(255, 255, 255, 0.08);
        border-radius: 12px;
        padding: 10px 14px;
        margin-bottom: 20px;
    ">
        {divider.join(step_items)}
    </div>
    """)
    st.markdown(html.strip(), unsafe_allow_html=True)


def page_header(title, subtitle):
    st.markdown(
        textwrap.dedent(f"""
        <div style="text-align: center; padding: 0.8rem 0 1.6rem;">
            <div class="brand-badge">
                <span class="pulse-dot"></span>
                Enterprise Security Gateway
            </div>
            <h1 class="gradient-title">{APP_NAME}</h1>
            <p class="gradient-subtext">{subtitle}</p>
            <div style="
                display: flex;
                align-items: center;
                justify-content: center;
                gap: 12px;
                margin-top: 18px;
            ">
                <span style="height: 1px; width: 36px; background: rgba(255, 255, 255, 0.12);"></span>
                <span style="
                    font-size: 0.85rem;
                    font-weight: 600;
                    color: #CBD5E1;
                    text-transform: uppercase;
                    letter-spacing: 0.08em;
                ">{title}</span>
                <span style="height: 1px; width: 36px; background: rgba(255, 255, 255, 0.12);"></span>
            </div>
        </div>
        """),
        unsafe_allow_html=True
    )


def clear_reset_flow():
    st.session_state.reset_email = ""
    st.session_state.reset_token = None
    st.session_state.reset_stage = "email"
    st.session_state.otp_verified = False
    st.session_state.pop("otp_notice", None)
    st.session_state.pop("last_otp_component_event", None)
    st.session_state.pop("last_reset_email_event", None)
    st.session_state.pop("reset_email_notice", None)
    st.session_state.pop("last_reset_password_event", None)
    st.session_state.pop("reset_password_notice", None)
    st.session_state.pop("advance_to_otp", None)
    st.session_state.pop("reset_question", None)
    st.session_state.pop("security_notice", None)
    st.session_state.pop("last_security_recovery_event", None)


def go(page):
    st.session_state.page = page
    st.rerun()


def apply_auth_card_theme():
    """Give native sign-in and sign-up widgets the recovery-card appearance."""
    st.markdown("""
    <style>
    .stApp,[data-testid="stAppViewContainer"]{background:#6352E0!important;background-image:linear-gradient(to right,rgba(255,255,255,.08) 1px,transparent 1px),linear-gradient(to bottom,rgba(255,255,255,.08) 1px,transparent 1px),radial-gradient(circle at 50% 50%,#7260EE,#523FD6)!important;background-size:80px 80px,80px 80px,100% 100%!important;background-attachment:fixed!important}
    .main .block-container,[data-testid="stMainBlockContainer"]{max-width:1280px!important;padding-top:2rem!important}
    body .stForm,body [data-testid="stForm"]{max-width:760px!important;margin-left:auto!important;margin-right:auto!important;background:#fff!important;border:0!important;border-radius:14px!important;padding:38px!important;box-shadow:0 20px 50px rgba(0,0,0,.18)!important}
    body .stForm label,body .stForm p,body [data-testid="stForm"] label,body [data-testid="stForm"] p{color:#334155!important;font-weight:600!important}
    body .stForm input,body .stForm select,body [data-testid="stForm"] input,body [data-testid="stForm"] select{background:#fff!important;color:#1e293b!important;border:1.5px solid #cbd5e1!important;border-radius:8px!important}
    body .stForm button,body [data-testid="stForm"] button,body [data-testid="stFormSubmitButton"] button{background:linear-gradient(90deg,#6260e6,#8343ad)!important;color:#fff!important;border:0!important;border-radius:8px!important;font-weight:700!important}
    [data-testid="stVerticalBlock"] > div > div > button{border-radius:8px!important}
    </style>
    """, unsafe_allow_html=True)


# --------------------------------------------------
# LOGIN PAGE
# --------------------------------------------------
def show_login():
    st.markdown("""<style>
    .stApp,[data-testid="stAppViewContainer"]{background:#6352E0!important;background-image:linear-gradient(to right,rgba(255,255,255,.08) 1px,transparent 1px),linear-gradient(to bottom,rgba(255,255,255,.08) 1px,transparent 1px),radial-gradient(circle at 50% 50%,#7260EE,#523FD6)!important;background-size:80px 80px,80px 80px,100% 100%!important;background-attachment:fixed!important}header[data-testid="stHeader"]{display:none!important}.main .block-container,[data-testid="stMainBlockContainer"]{max-width:100%!important;width:100%!important;padding:1.5rem 0!important}iframe[data-testid="stCustomComponentV1"]{width:100%!important;border:0!important}
    </style>""", unsafe_allow_html=True)
    notice = st.session_state.pop("login_notice", None)
    event = login_component(notice=notice, key="login_component")
    if event:
        event_id = event.get("event_id")
        if event_id and event_id == st.session_state.get("last_login_event"):
            return
        st.session_state.last_login_event = event_id
        action = event.get("action")
        if action == "signup":
            go("Signup")
        if action == "forgot":
            clear_reset_flow()
            go("Forgot")
        if action == "login":
            login_id = str(event.get("email", "")).strip()
            password = str(event.get("password", ""))

            if not login_id:
                st.session_state.login_notice = {"kind": "error", "text": "Please enter your username or email address.", "field": "email"}
                st.rerun()

            if not password:
                st.session_state.login_notice = {"kind": "error", "text": "Please enter your password.", "field": "password"}
                st.rerun()

            remaining = login_lockout_remaining(login_id)
            if remaining:
                st.session_state.login_notice = {"kind": "error", "text": f"Too many failed attempts. Try again in {remaining} seconds."}
                st.rerun()

            user = get_user(login_id)
            if user:
                remaining_user = login_lockout_remaining(user["email"])
                if remaining_user:
                    st.session_state.login_notice = {"kind": "error", "text": f"Too many failed attempts. Try again in {remaining_user} seconds."}
                    st.rerun()

            if user and check_password(password, user["password_hash"]):
                clear_failed_logins(login_id)
                clear_failed_logins(user["email"])
                st.session_state.login_token = issue_login_token(user["email"])
                go("Dashboard")
            else:
                record_failed_login(login_id)
                if user:
                    record_failed_login(user["email"])
                st.session_state.login_notice = {"kind": "error", "text": "Invalid username/email or password.", "field": "both"}
                st.rerun()
    return

    page_header(
        "Sign in to your account",
        "Encrypted authentication with session security"
    )

    with st.form("login_form"):
        email = st.text_input(
            "Email Address",
            placeholder="you@company.com"
        ).lower().strip()

        password = st.text_input(
            "Password",
            type="password",
            placeholder="••••••••••••"
        )

        st.markdown("<div style='height: 6px;'></div>", unsafe_allow_html=True)

        submitted = st.form_submit_button(
            "Sign In to Portal →",
            use_container_width=True
        )

    if submitted:
        if not email or not password:
            st.warning(
                "Please enter both your email address and password."
            )

        elif not JWT_SECRET:
            st.error(
                "JWT_SECRET is not configured. "
                "Check your Google Colab Secrets (key icon on left) and restart."
            )

        else:
            remaining = login_lockout_remaining(email)

            if remaining:
                st.error(
                    f"Too many failed attempts. "
                    f"Try again in about {remaining} seconds."
                )

            else:
                user = get_user(email)

                if (
                    user
                    and check_password(
                        password,
                        user["password_hash"]
                    )
                ):
                    clear_failed_logins(email)

                    st.session_state.login_token = (
                        issue_login_token(email)
                    )

                    go("Dashboard")

                else:
                    record_failed_login(email)
                    st.error("Invalid email or password.")

    st.markdown("<div style='height: 8px;'></div>", unsafe_allow_html=True)
    col1, col2 = st.columns(2)

    with col1:
        if st.button(
            "✨ Create account",
            use_container_width=True
        ):
            go("Signup")

    with col2:
        if st.button(
            "🔑 Forgot password?",
            use_container_width=True
        ):
            clear_reset_flow()
            go("Forgot")


# --------------------------------------------------
# SIGNUP PAGE
# --------------------------------------------------
def show_signup():
    st.markdown("""<style>
    .stApp,[data-testid="stAppViewContainer"]{background:#6352E0!important;background-image:linear-gradient(to right,rgba(255,255,255,.08) 1px,transparent 1px),linear-gradient(to bottom,rgba(255,255,255,.08) 1px,transparent 1px),radial-gradient(circle at 50% 50%,#7260EE,#523FD6)!important;background-size:80px 80px,80px 80px,100% 100%!important;background-attachment:fixed!important}header[data-testid="stHeader"]{display:none!important}.main .block-container,[data-testid="stMainBlockContainer"]{max-width:100%!important;width:100%!important;padding:1.5rem 0!important}iframe[data-testid="stCustomComponentV1"]{width:100%!important;border:0!important}
    </style>""", unsafe_allow_html=True)
    apply_auth_card_theme()
    # New accounts must verify ownership of their email before registration.
    if st.session_state.get("signup_stage") == "complete":
        # Show the success "Account Created" page.
        pending = st.session_state.get("pending_signup", {})
        event = signup_complete_component(
            email=pending.get("email", ""), key="signup_complete_comp"
        )
        if event:
            event_id = event.get("event_id")
            if event_id != st.session_state.get("last_signup_complete_event"):
                st.session_state.last_signup_complete_event = event_id
                if event.get("action") == "go_login":
                    for key in ("signup_stage", "pending_signup", "signup_token",
                                "last_signup_otp_event", "last_signup_complete_event"):
                        st.session_state.pop(key, None)
                    go("Login")
        return

    if st.session_state.get("signup_stage") == "otp":
        pending = st.session_state.get("pending_signup", {})
        notice = st.session_state.pop("otp_notice", None)
        event = signup_otp_component(
            email=pending.get("email", ""),
            notice=notice,
            key="signup_otp_comp"
        )
        if not event:
            return
        event_id = event.get("event_id")
        if event_id == st.session_state.get("last_signup_otp_event"):
            return
        st.session_state.last_signup_otp_event = event_id
        action = event.get("action")
        if action == "verify_otp":
            ok, message = verify_otp_token(
                st.session_state.get("signup_token"),
                str(event.get("otp_code", "")),
                pending.get("email", ""),
            )
            if ok:
                created, reg_message = register_user(**pending)
                if created:
                    st.session_state.signup_stage = "complete"
                    st.rerun()
                else:
                    st.session_state.otp_notice = {"kind": "error", "text": reg_message}
                    st.rerun()
            else:
                if "invalidated" in message.lower():
                    st.session_state.signup_token = None
                st.session_state.otp_notice = {"kind": "error", "text": message}
                st.rerun()
        elif action == "resend_otp":
            otp_value = generate_otp()
            ok, message = send_otp_email(pending.get("email", ""), otp_value, purpose="signup")
            if ok:
                st.session_state.signup_token = make_otp_token(pending["email"], otp_value)
                st.session_state.otp_notice = {"kind": "success", "text": "A new code has been sent."}
            else:
                st.session_state.otp_notice = {"kind": "error", "text": message}
            st.rerun()
        elif action == "go_back":
            for key in ("signup_stage", "pending_signup", "signup_token", "last_signup_otp_event"):
                st.session_state.pop(key, None)
            go("Signup")
        return

    notice = st.session_state.pop("signup_notice", None)
    event = signup_component(notice=notice, key="signup_component")
    if not event:
        return
    event_id = event.get("event_id")
    if event_id and event_id == st.session_state.get("last_signup_event"):
        return
    st.session_state.last_signup_event = event_id
    if event.get("action") == "signin":
        go("Login")
    if event.get("action") == "signup":
        username, email = str(event.get("username", "")).strip(), str(event.get("email", "")).lower().strip()
        password, confirm = str(event.get("password", "")), str(event.get("confirm", ""))
        question = str(event.get("question", "")).strip()
        answer = str(event.get("answer", "")).strip()

        # Pinpoint 1: Username validation
        if not username:
            st.session_state.signup_notice = {"kind": "error", "text": "Please enter a username / full name.", "field": "username"}
            st.rerun()
        if len(username) < 3:
            st.session_state.signup_notice = {"kind": "error", "text": "Username must be at least 3 characters long.", "field": "username"}
            st.rerun()
        if get_user_by_username(username):
            st.session_state.signup_notice = {"kind": "error", "text": "This username is already taken. Please choose another.", "field": "username"}
            st.rerun()

        # Pinpoint 2: Email validation
        if not email:
            st.session_state.signup_notice = {"kind": "error", "text": "Please enter your email address.", "field": "email"}
            st.rerun()
        if not email_is_valid(email):
            st.session_state.signup_notice = {
                "kind": "error",
                "text": "Invalid email address format (must be like name@example.com with 2+ letters before and after dots).",
                "field": "email"
            }
            st.rerun()
        if get_user(email):
            st.session_state.signup_notice = {"kind": "error", "text": "An account with this email address already exists. Please sign in.", "field": "email"}
            st.rerun()

        # Pinpoint 3: Password validation
        if not password:
            st.session_state.signup_notice = {"kind": "error", "text": "Please enter a password.", "field": "password"}
            st.rerun()
        _, _, _, _, checks = check_password_strength(password)
        if not checks["length"]:
            st.session_state.signup_notice = {"kind": "error", "text": "Password must be at least 8 characters long.", "field": "password"}
            st.rerun()
        if not (checks["uppercase"] and checks["lowercase"]):
            st.session_state.signup_notice = {"kind": "error", "text": "Password must contain both uppercase and lowercase letters.", "field": "password"}
            st.rerun()
        if not checks["digit"]:
            st.session_state.signup_notice = {"kind": "error", "text": "Password must contain at least one number (0-9).", "field": "password"}
            st.rerun()
        if not checks["special"]:
            st.session_state.signup_notice = {"kind": "error", "text": "Password must contain at least one special symbol (!@#$%^&*).", "field": "password"}
            st.rerun()

        # Pinpoint 4: Confirm password validation
        if not confirm:
            st.session_state.signup_notice = {"kind": "error", "text": "Please confirm your password.", "field": "confirm"}
            st.rerun()
        if password != confirm:
            st.session_state.signup_notice = {"kind": "error", "text": "Confirm Password does not match Password.", "field": "confirm"}
            st.rerun()

        # Pinpoint 5: Security Question validation
        if not question:
            st.session_state.signup_notice = {"kind": "error", "text": "Please select a security question from the list.", "field": "question"}
            st.rerun()

        # Pinpoint 6: Security Answer validation
        ans_ok, ans_msg = security_answer_is_valid(answer)
        if not ans_ok:
            st.session_state.signup_notice = {"kind": "error", "text": ans_msg, "field": "answer"}
            st.rerun()

        otp_value = generate_otp()
        ok, message = send_otp_email(email, otp_value, purpose="signup")
        if ok:
            st.session_state.pending_signup = {"username": username, "email": email, "password": password, "question": question, "answer": answer}
            st.session_state.signup_token = make_otp_token(email, otp_value)
            st.session_state.signup_stage = "otp"
            st.rerun()
        else:
            st.session_state.signup_notice = {"kind": "error", "text": message or "Failed to send verification code. Please try again."}
            st.rerun()
    return

    page_header(
        "Create an account",
        "Register for secure enterprise portal access"
    )

    with st.form("signup_form"):
        username = st.text_input(
            "Full Name",
            placeholder="John Doe"
        )

        email = st.text_input(
            "Email Address",
            placeholder="you@company.com"
        ).lower().strip()

        password = st.text_input(
            "Password",
            type="password",
            placeholder="At least 8 characters with mix of case, numbers & symbols",
            help="Must be at least 8 characters."
        )

        confirm = st.text_input(
            "Confirm Password",
            type="password",
            placeholder="Re-enter password"
        )

        question = st.selectbox(
            "Security Question (Required)",
            [
                "What was the name of your first pet?",
                "What is your favourite city?",
                "What was your first school?",
                "What was the make and model of your first car?",
                "What is your mother's maiden name?",
                "In what city or town was your first job?",
                "What was the name of the street you grew up on?",
                "What was the name of your childhood best friend?",
                "What is your favorite book or movie?"
            ]
        )

        answer = st.text_input(
            "Security Answer",
            placeholder="Your secret answer"
        )

        render_password_meter(password if "password" in locals() and password else "")

        submitted = st.form_submit_button(
            "Create Account →",
            use_container_width=True
        )

    if submitted:
        score, level, color, percent, checks = check_password_strength(password)

        if not username.strip() or not email or not password:
            st.warning(
                "Please complete your name, email, and password."
            )

        elif not email_is_valid(email):
            st.warning("Enter a valid email address.")

        elif len(password) < 8:
            st.warning(
                "Password must be at least 8 characters long."
            )

        elif not (checks["digit"] or checks["special"]):
            st.warning(
                "Password should include at least one number or special character for security."
            )

        elif password != confirm:
            st.error("Passwords do not match.")

        elif not question or not answer.strip():
            st.warning("Please choose a security question and enter an answer.")

        else:
            # Defer user creation until the OTP is verified.
            otp_value = generate_otp()
            ok, message = send_otp_email(email, otp_value, purpose="signup")
            if ok:
                st.session_state.pending_signup = {
                    "username": username, "email": email, "password": password,
                    "question": question, "answer": answer,
                }
                st.session_state.signup_token = make_otp_token(email, otp_value)
                st.session_state.signup_stage = "otp"
                st.rerun()
            else:
                st.error(message)

    st.markdown("<div style='height: 8px;'></div>", unsafe_allow_html=True)
    if st.button(
        "← Back to sign in",
        use_container_width=True
    ):
        go("Login")


# --------------------------------------------------
# FORGOT PASSWORD / OTP RESET PAGE
# --------------------------------------------------
def show_forgot():
    if not JWT_SECRET:
        page_header("Reset your password", "Secure account recovery")
        st.error(
            "JWT_SECRET is not configured. "
            "Add it to Google Colab Secrets (key icon on left) and restart."
        )
        if st.button("Back to sign in"):
            go("Login")
        return

    # Complete a successful email-send transition before rendering either
    # recovery component. This avoids the email component being rendered again
    # after Streamlit processes its event callback.
    if st.session_state.pop("advance_to_otp", False):
        st.session_state.reset_stage = "otp"

    # Stage 1: Request OTP
    if st.session_state.reset_stage == "email":
        st.markdown("""
        <style>
        .stApp, [data-testid="stAppViewContainer"], [data-testid="stHeader"] {
            background-color: #6352E0 !important;
            background-image: linear-gradient(to right, rgba(255,255,255,.08) 1px, transparent 1px), linear-gradient(to bottom, rgba(255,255,255,.08) 1px, transparent 1px), radial-gradient(circle at 50% 50%, #7260EE 0%, #523FD6 100%) !important;
            background-size: 80px 80px, 80px 80px, 100% 100% !important; background-attachment: fixed !important;
        }
        header[data-testid="stHeader"] { display:none !important; }
        .main .block-container, [data-testid="stMainBlockContainer"] { max-width:100% !important; width:100% !important; padding:2rem 0 !important; }
        iframe[data-testid="stCustomComponentV1"] { width:100% !important; border:none !important; }
        </style>
        """, unsafe_allow_html=True)

        event = reset_email_component(
            notice=st.session_state.get("reset_email_notice"),
            key="reset_email_component",
        )
        if event:
            event_id = event.get("event_id")
            if event_id and event_id == st.session_state.get("last_reset_email_event"):
                return
            st.session_state.last_reset_email_event = event_id
            identifier = str(event.get("email", "")).strip()
            if event.get("action") == "signin":
                clear_reset_flow()
                go("Login")
            elif event.get("action") == "security_question":
                if not identifier:
                    st.session_state.reset_email_notice = {"kind": "error", "text": "Please enter your username or email address.", "field": "email"}
                    st.rerun()
                user = get_user(identifier)
                if not user:
                    st.session_state.reset_email_notice = {"kind": "error", "text": "No registered account found matching that username or email.", "field": "email"}
                    st.rerun()
                if not user["security_question"] or not user["security_answer_hash"]:
                    st.session_state.reset_email_notice = {"kind": "error", "text": "Security-question recovery is not available for this account. Use email verification instead.", "field": "email"}
                    st.rerun()
                st.session_state.reset_email = user["email"]
                st.session_state.reset_question = user["security_question"]
                st.session_state.reset_stage = "security_question"
                st.session_state.pop("reset_email_notice", None)
                st.rerun()
            elif event.get("action") == "request_otp":
                if not identifier:
                    st.session_state.reset_email_notice = {"kind": "error", "text": "Please enter your registered email address or username.", "field": "email"}
                    st.rerun()
                user = get_user(identifier)
                target_email = user["email"] if user else (identifier.lower() if email_is_valid(identifier) else None)
                if not target_email or not email_is_valid(target_email):
                    st.session_state.reset_email_notice = {
                        "kind": "error", "text": "Please enter a valid registered email address or username.", "field": "email"
                    }
                elif not user:
                    st.session_state.reset_email_notice = {
                        "kind": "error", "text": "No registered account found matching that username or email.", "field": "email"
                    }
                elif not can_request_otp(target_email):
                    st.session_state.reset_email_notice = {
                        "kind": "error", "text": "Too many code requests. Please wait before trying again."
                    }
                else:
                    otp_value = generate_otp()
                    ok, message = send_otp_email(target_email, otp_value, purpose="reset")
                    if ok:
                        record_otp_request(target_email)
                        st.session_state.reset_email = target_email
                        st.session_state.reset_token = make_otp_token(target_email, otp_value)
                        st.session_state.advance_to_otp = True
                        st.session_state.pop("reset_email_notice", None)
                    else:
                        st.session_state.reset_email_notice = {"kind": "error", "text": message}
                st.rerun()
        return

        page_header(
            "Reset your password",
            "Verify your email with a one-time cryptographic code"
        )
        render_step_indicator("email")

        st.markdown(
            textwrap.dedent("""
            <div style="
                background: rgba(15, 23, 42, 0.6);
                border: 1px solid rgba(255, 255, 255, 0.08);
                border-radius: 14px;
                padding: 16px 18px;
                margin-bottom: 18px;
                display: flex;
                align-items: center;
                gap: 12px;
            ">
                <span style="font-size: 26px;">✉️</span>
                <div style="font-size: 0.88rem; color: #94A3B8;">
                    Enter your registered email address. We will dispatch a 6-digit verification code to your inbox.
                </div>
            </div>
            """),
            unsafe_allow_html=True
        )

        with st.form("request_otp_form"):
            email = st.text_input(
                "Registered Email Address",
                placeholder="you@company.com"
            ).lower().strip()

            submitted = st.form_submit_button(
                "Send Verification Code →",
                use_container_width=True
            )

        if submitted:
            if not email_is_valid(email):
                st.warning("Enter a valid email address.")

            elif not get_user(email):
                st.info(
                    "If the address is registered and email "
                    "is configured, a code has been dispatched."
                )

            elif not can_request_otp(email):
                st.error(
                    "Too many code requests. "
                    "Please wait 15 minutes before trying again."
                )

            else:
                otp_value = generate_otp()
                ok, message = send_otp_email(email, otp_value, purpose="reset")

                if ok:
                    record_otp_request(email)
                    st.session_state.reset_email = email
                    st.session_state.reset_token = make_otp_token(email, otp_value)
                    st.session_state.reset_stage = "otp"
                    st.success("Verification code sent. Check your inbox.")
                    st.rerun()
                else:
                    st.error(message)

        st.markdown("<div style='height: 8px;'></div>", unsafe_allow_html=True)
        if st.button("← Back to sign in", use_container_width=True):
            clear_reset_flow()
            go("Login")

    # Stage 1b: Security question — an alternative to receiving an email OTP.
    elif st.session_state.reset_stage == "security_question":
        st.markdown("""<style>
        .stApp,[data-testid="stAppViewContainer"]{background:#6352E0!important;background-image:linear-gradient(to right,rgba(255,255,255,.08) 1px,transparent 1px),linear-gradient(to bottom,rgba(255,255,255,.08) 1px,transparent 1px),radial-gradient(circle at 50% 50%,#7260EE,#523FD6)!important;background-size:80px 80px,80px 80px,100% 100%!important;background-attachment:fixed!important}
        header[data-testid="stHeader"]{display:none!important}.main .block-container,[data-testid="stMainBlockContainer"]{max-width:100%!important;width:100%!important;padding:2rem 0!important}iframe[data-testid="stCustomComponentV1"]{width:100%!important;border:0!important}
        </style>""", unsafe_allow_html=True)
        email = st.session_state.get("reset_email", "")

        # Check if already locked out
        lockout = get_security_lockout_remaining(email)
        if lockout > 0:
            st.session_state.security_notice = {
                "kind": "error",
                "text": f"Too many failed attempts. Security recovery locked. Try again in {lockout} seconds.",
                "field": "answer",
                "locked": True,
                "ts": time.time()
            }

        event = security_recovery_component(
            email=email,
            question=st.session_state.get("reset_question", ""),
            notice=st.session_state.get("security_notice"),
            key="security_recovery_component",
        )
        if not event:
            return
        event_id = event.get("event_id")
        if event_id and event_id == st.session_state.get("last_security_recovery_event"):
            return
        st.session_state.last_security_recovery_event = event_id
        action = event.get("action")
        if action == "signin":
            clear_reset_flow()
            go("Login")
        if action == "back":
            st.session_state.reset_stage = "email"
            st.session_state.pop("security_notice", None)
            st.rerun()
        if action == "verify_answer":
            lockout = get_security_lockout_remaining(email)
            if lockout > 0:
                st.session_state.security_notice = {
                    "kind": "error",
                    "text": f"Too many failed attempts. Security recovery locked. Try again in {lockout} seconds.",
                    "field": "answer",
                    "locked": True,
                    "ts": time.time()
                }
                st.rerun()

            user = get_user(email)
            answer = str(event.get("answer", "")).strip().lower()
            if not answer:
                st.session_state.security_notice = {
                    "kind": "error",
                    "text": "Please enter your security answer.",
                    "field": "answer",
                    "locked": False,
                    "ts": time.time()
                }
                st.rerun()
            ans_ok, ans_msg = security_answer_is_valid(answer)
            if not ans_ok:
                st.session_state.security_notice = {
                    "kind": "error",
                    "text": ans_msg,
                    "field": "answer",
                    "locked": False,
                    "ts": time.time()
                }
                st.rerun()
            if user and user["security_answer_hash"] and check_password(answer, user["security_answer_hash"]):
                clear_security_attempts(email)
                st.session_state.otp_verified = True
                st.session_state.reset_stage = "password"
                st.session_state.pop("security_notice", None)
            else:
                attempts = record_failed_security_attempt(email)
                rem = max(0, MAX_SECURITY_ANSWER_ATTEMPTS - attempts)
                if rem > 0:
                    plural = "attempt" if rem == 1 else "attempts"
                    text = f"That security answer is incorrect. You have {rem} {plural} remaining."
                    st.session_state.security_notice = {
                        "kind": "error",
                        "text": text,
                        "field": "answer",
                        "locked": False,
                        "ts": time.time()
                    }
                else:
                    mins = max(1, SECURITY_LOCKOUT_SECONDS // 60)
                    text = f"Too many failed attempts. Account recovery locked for {mins} minutes."
                    st.session_state.security_notice = {
                        "kind": "error",
                        "text": text,
                        "field": "answer",
                        "locked": True,
                        "ts": time.time()
                    }
            st.rerun()
        return

    # Stage 2: Verify OTP — Pixel-Exact Full Width Component with Action Bridges
    elif st.session_state.reset_stage == "otp":
        email_display = st.session_state.reset_email or "example@email.com"

        # Kept only as an inactive fallback while the visual component below is
        # used. The component now sends actions through Streamlit's supported
        # message protocol, preserving the approved card layout.
        if False:
            # These controls intentionally live in the Streamlit page rather than in
        # the embedded HTML preview below.  ``components.html`` runs inside a
        # sandboxed iframe, so it cannot reliably update the parent page's URL
        # or session state.  Native controls keep every recovery action on the
        # same server-side reset flow.
            page_header("Verify your email", "Enter the verification code we sent")
            render_step_indicator("otp")
            st.markdown(
            textwrap.dedent(f"""
            <div style="background:#FFFFFF; color:#1E293B; border-radius:14px;
                        padding:26px; box-shadow:0 20px 50px rgba(0,0,0,.18);
                        margin-bottom:18px; text-align:center;">
                <div style="font-size:1.2rem; font-weight:700; margin-bottom:8px;">
                    Verify
                </div>
                <div style="color:#64748B; font-size:.92rem;">
                    We sent a 6-digit verification code to
                    <strong style="color:#6260E6;">{email_display}</strong>.
                </div>
            </div>
            """),
            unsafe_allow_html=True,
            )

            with st.form("verify_otp_form"):
                code = st.text_input(
                "6-digit verification code",
                max_chars=6,
                placeholder="123456",
                ).strip()
                verify_submitted = st.form_submit_button(
                "✓ Verify", use_container_width=True
                )

            if verify_submitted:
                if not (code.isdigit() and len(code) == 6):
                    st.error("Enter the 6-digit code from your email.")
                else:
                    ok, message = verify_otp_token(
                        st.session_state.reset_token,
                        code,
                        st.session_state.reset_email,
                    )
                    if ok:
                        st.session_state.otp_verified = True
                        st.session_state.reset_stage = "password"
                        st.rerun()
                    st.error(message)

            resend_col, change_email_col = st.columns(2)
            with resend_col:
                if st.button("↻ Resend Code", use_container_width=True):
                    if not can_request_otp(st.session_state.reset_email):
                        st.error("Too many code requests. Please wait before trying again.")
                    else:
                        otp_value = generate_otp()
                        ok, message = send_otp_email(
                            st.session_state.reset_email, otp_value, purpose="reset"
                        )
                        if ok:
                            record_otp_request(st.session_state.reset_email)
                            st.session_state.reset_token = make_otp_token(
                                st.session_state.reset_email, otp_value
                            )
                            st.success("A new verification code has been sent.")
                        else:
                            st.error(message)
            with change_email_col:
                if st.button("📝 Change Email Address", use_container_width=True):
                    clear_reset_flow()
                    st.rerun()

            if st.button("Already verified? Sign in to your account", use_container_width=True):
                clear_reset_flow()
                go("Login")

        # Apply global CSS to remove dark borders and enforce widescreen canvas
        st.markdown(textwrap.dedent("""
        <style>
        .stApp, [data-testid="stAppViewContainer"], [data-testid="stHeader"] {
            background-color: #6352E0 !important;
            background-image: 
                linear-gradient(to right, rgba(255, 255, 255, 0.08) 1px, transparent 1px),
                linear-gradient(to bottom, rgba(255, 255, 255, 0.08) 1px, transparent 1px),
                radial-gradient(circle at 50% 50%, #7260EE 0%, #523FD6 100%) !important;
            background-size: 80px 80px, 80px 80px, 100% 100% !important;
            background-attachment: fixed !important;
        }

        header[data-testid="stHeader"] { display: none !important; }

        .main .block-container,
        [data-testid="stMainBlockContainer"] {
            max-width: 100% !important;
            width: 100% !important;
            padding: 2rem 0 !important;
        }

        iframe[data-testid="stCustomComponentV1"] {
            width: 100% !important;
            border: none !important;
        }
        </style>
        """), unsafe_allow_html=True)

        # Trigger hidden native Streamlit buttons via Python state handlers
        col_hidden1, col_hidden2, col_hidden3 = st.columns([1, 1, 1])
        with col_hidden1:
            trigger_verify = st.hidden_input(key="hidden_verify") if hasattr(st, "hidden_input") else None

        html_code = f"""
        <!DOCTYPE html>
        <html>
        <head>
          <meta charset="utf-8">
          <style>
            * {{
              box-sizing: border-box;
              font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
            }}
            html, body {{
              margin: 0;
              padding: 20px;
              width: 100%;
              min-height: 100vh;
              background-color: transparent;
              display: flex;
              justify-content: center;
              align-items: center;
            }}

            .card {{
              background: #FFFFFF;
              width: 100%;
              max-width: 860px;
              border-radius: 14px;
              padding: 45px 75px;
              box-shadow: 0 20px 50px rgba(0, 0, 0, 0.18);
              text-align: center;
            }}

            .stepper {{
              display: flex;
              align-items: center;
              justify-content: center;
              max-width: 320px;
              margin: 0 auto 30px auto;
            }}
            .step {{
              display: flex;
              flex-direction: column;
              align-items: center;
              min-width: 60px;
            }}
            .circle {{
              width: 32px;
              height: 32px;
              border-radius: 50%;
              display: flex;
              align-items: center;
              justify-content: center;
              font-size: 13px;
              font-weight: 700;
            }}
            .step-1 .circle {{ background: #00B982; color: #FFF; }}
            .step-2 .circle {{ background: #6260E6; color: #FFF; }}
            .step-3 .circle {{ background: #F1F5F9; color: #94A3B8; }}
            .step-title {{
              font-size: 11px;
              font-weight: 600;
              margin-top: 6px;
            }}
            .step-1 .step-title {{ color: #00B982; }}
            .step-2 .step-title {{ color: #6260E6; }}
            .step-3 .step-title {{ color: #94A3B8; }}
            .line {{
              height: 2px;
              flex: 1;
              margin: 0 8px 18px 8px;
            }}
            .line-active {{ background: #00B982; }}
            .line-inactive {{ background: #E2E8F0; }}

            .icon-badge {{
              width: 58px;
              height: 58px;
              border-radius: 50%;
              background: linear-gradient(135deg, #6260E6, #8343AD);
              display: flex;
              align-items: center;
              justify-content: center;
              margin: 0 auto 18px auto;
              box-shadow: 0 8px 20px rgba(98, 96, 230, 0.35);
            }}

            h2 {{
              font-size: 24px;
              font-weight: 700;
              color: #1E293B;
              margin: 0 0 8px 0;
            }}
            p.subtext {{
              font-size: 13px;
              color: #64748B;
              margin: 0;
              line-height: 1.5;
            }}
            p.subtext strong {{ color: #6260E6; font-weight: 500; }}

            .enter-label {{
              font-size: 13px;
              font-weight: 600;
              color: #334155;
              margin-top: 24px;
              margin-bottom: 14px;
            }}

            .otp-container {{
              display: flex;
              justify-content: center;
              gap: 12px;
              margin-bottom: 14px;
            }}
            .otp-input {{
              width: 52px;
              height: 54px;
              background-color: #E8FAF4 !important;
              border: 1.8px solid #00B982 !important;
              border-radius: 8px;
              text-align: center;
              font-size: 22px;
              font-weight: 700;
              color: #1E293B;
              outline: none;
            }}
            .otp-input:focus {{
              border-color: #008F64 !important;
              background-color: #FFFFFF !important;
            }}

            .spam-note {{
              font-size: 12px;
              color: #94A3B8;
              margin-bottom: 14px;
            }}

            .resend-text {{
              font-size: 13px;
              color: #64748B;
              margin-bottom: 20px;
            }}
            .resend-link {{
              color: #6260E6;
              font-weight: 600;
              cursor: pointer;
              text-decoration: none;
            }}
            .resend-link:hover {{ text-decoration: underline; }}
            .resend-link.disabled {{
              color: #94A3B8;
              cursor: not-allowed;
              pointer-events: none;
              text-decoration: none;
            }}

            .btn-verify {{
              width: 100%;
              max-width: 280px;
              height: 44px;
              background: linear-gradient(90deg, #6260E6 0%, #8343AD 100%);
              color: #FFFFFF;
              border: none;
              border-radius: 8px;
              font-size: 14px;
              font-weight: 600;
              cursor: pointer;
              box-shadow: 0 4px 14px rgba(96, 93, 236, 0.3);
              margin: 0 auto;
              display: flex;
              align-items: center;
              justify-content: center;
              gap: 8px;
            }}

            .divider {{
              display: flex;
              align-items: center;
              gap: 16px;
              margin: 26px 0 20px 0;
              color: #CBD5E1;
              font-size: 12px;
            }}
            .divider-line {{
              height: 1px;
              background: #E2E8F0;
              flex: 1;
            }}

            .trouble-actions {{
              display: flex;
              justify-content: center;
              gap: 16px;
              margin-bottom: 18px;
            }}
            .pill-btn {{
              background: #FFFFFF;
              border: 1px solid #E2E8F0;
              border-radius: 8px;
              padding: 9px 18px;
              font-size: 13px;
              color: #475569;
              font-weight: 500;
              cursor: pointer;
              display: flex;
              align-items: center;
              gap: 6px;
              text-decoration: none;
              box-shadow: 0 1px 3px rgba(0,0,0,0.04);
            }}
            .pill-btn:hover {{
              background: #F8FAFC;
              border-color: #CBD5E1;
            }}

            .signin-text {{
              font-size: 13px;
              color: #64748B;
            }}
            .signin-link {{
              color: #6260E6;
              font-weight: 600;
              text-decoration: none;
              cursor: pointer;
            }}
            .signin-link:hover {{ text-decoration: underline; }}
          </style>
        </head>
        <body>

          <div class="card">
            <div class="stepper">
              <div class="step step-1">
                <div class="circle">1</div>
                <div class="step-title">Account</div>
              </div>
              <div class="line line-active"></div>
              <div class="step step-2">
                <div class="circle">2</div>
                <div class="step-title">Verify</div>
              </div>
              <div class="line line-inactive"></div>
              <div class="step step-3">
                <div class="circle">3</div>
                <div class="step-title">Complete</div>
              </div>
            </div>

            <div class="icon-badge">
              <svg width="26" height="26" viewBox="0 0 24 24" fill="none" stroke="white" stroke-width="2" stroke-linecap="round" stroke-linejoin="round">
                <path d="M4 4h16c1.1 0 2 .9 2 2v12c0 1.1-.9 2-2 2H4c-1.1 0-2-.9-2-2V6c0-1.1.9-2 2-2z"></path>
                <polyline points="22,6 12,13 2,6"></polyline>
              </svg>
            </div>

            <h2>Verify</h2>
            <p class="subtext">
              We sent a verification code to <strong>{email_display}</strong><br>
              Enter the 6-digit code to continue
            </p>

            <div class="enter-label">Enter Code</div>

            <form id="otpForm" onsubmit="submitOtp(event)">
              <div class="otp-container">
                <input type="text" class="otp-input" maxlength="1" id="digit1" autofocus onkeyup="moveFocus(this, 'digit2', null)">
                <input type="text" class="otp-input" maxlength="1" id="digit2" onkeyup="moveFocus(this, 'digit3', 'digit1')">
                <input type="text" class="otp-input" maxlength="1" id="digit3" onkeyup="moveFocus(this, 'digit4', 'digit2')">
                <input type="text" class="otp-input" maxlength="1" id="digit4" onkeyup="moveFocus(this, 'digit5', 'digit3')">
                <input type="text" class="otp-input" maxlength="1" id="digit5" onkeyup="moveFocus(this, 'digit6', 'digit4')">
                <input type="text" class="otp-input" maxlength="1" id="digit6" onkeyup="moveFocus(this, null, 'digit5')">
              </div>

              <div class="spam-note">Check your spam folder if you don't see the email</div>

              <div class="resend-text">
                Didn't receive the code? 
                <a id="resendBtn" class="resend-link disabled" onclick="handleResend()">
                  &#8635; Resend Code (<span id="timer">30</span>s)
                </a>
              </div>

              <button type="submit" class="btn-verify">
                <span>&#10003;</span> Verify
              </button>
            </form>

            <div class="divider">
              <div class="divider-line"></div>
              <span>having trouble?</span>
              <div class="divider-line"></div>
            </div>

            <div class="trouble-actions">
              <button class="pill-btn" type="button" onclick="triggerAction('change_email')">&#128221; Change Email Address</button>
              <button class="pill-btn" type="button" onclick="alert('Support Assistance: support@infosysportal.com')">&#127911; Contact Support</button>
            </div>

            <div class="signin-text">
              Already verified? <a class="signin-link" onclick="triggerAction('signin')">Sign in to your account</a>
            </div>
          </div>

          <script>
            var countdown = 30;
            var timerElement = document.getElementById('timer');
            var resendBtn = document.getElementById('resendBtn');

            function startResendTimer() {{
              var interval = setInterval(function() {{
                countdown--;
                if (timerElement) {{
                  timerElement.innerText = countdown;
                }}
                if (countdown <= 0) {{
                  clearInterval(interval);
                  resendBtn.classList.remove('disabled');
                  resendBtn.innerHTML = '&#8635; Resend Code';
                }}
              }}, 1000);
            }}

            function handleResend() {{
              if (!resendBtn.classList.contains('disabled')) {{
                triggerAction('resend_otp');
              }}
            }}

            window.onload = function() {{
              startResendTimer();
            }};

            function moveFocus(current, nextId, prevId) {{
              if (current.value.length >= 1 && nextId) {{
                document.getElementById(nextId).focus();
              }}
              if (event.key === "Backspace" && prevId && current.value.length === 0) {{
                document.getElementById(prevId).focus();
              }}
            }}

            function submitOtp(e) {{
              e.preventDefault();
              var code = [
                document.getElementById('digit1').value,
                document.getElementById('digit2').value,
                document.getElementById('digit3').value,
                document.getElementById('digit4').value,
                document.getElementById('digit5').value,
                document.getElementById('digit6').value
              ].join('');

              var form = document.createElement('form');
              form.method = 'POST';
              form.action = window.top.location.href;
              
              var inputAction = document.createElement('input');
              inputAction.type = 'hidden';
              inputAction.name = 'component_action';
              inputAction.value = 'verify_otp';
              form.appendChild(inputAction);

              var inputCode = document.createElement('input');
              inputCode.type = 'hidden';
              inputCode.name = 'otp_code';
              inputCode.value = code;
              form.appendChild(inputCode);

              document.body.appendChild(form);
              window.top.location.search = '?action=verify_otp&otp_code=' + code;
            }}

            function triggerAction(actionName) {{
              window.top.location.search = '?action=' + actionName;
            }}
          </script>
        </body>
        </html>
        """

        event = otp_verification_component(
            email=email_display,
            notice=st.session_state.get("otp_notice"),
            key="otp_verification",
        )
        if not event:
            return

        event_id = event.get("event_id")
        if event_id and event_id == st.session_state.get("last_otp_component_event"):
            return
        st.session_state.last_otp_component_event = event_id
        action = event.get("action")
        if action == "verify_otp":
            code = str(event.get("otp_code", "")).strip()
            if not (code.isdigit() and len(code) == 6):
                st.session_state.otp_notice = {
                    "kind": "error", "text": "Please enter the complete 6-digit verification code.", "field": "otp"
                }
                st.rerun()
            ok, message = verify_otp_token(
                st.session_state.reset_token, code, st.session_state.reset_email
            )
            if ok:
                st.session_state.otp_verified = True
                st.session_state.reset_stage = "password"
                st.rerun()
            else:
                if "invalidated" in message.lower():
                    st.session_state.reset_token = None
                st.session_state.otp_notice = {"kind": "error", "text": message, "field": "otp"}
                st.rerun()
        elif action == "resend_otp":
            if not can_request_otp(st.session_state.reset_email):
                st.session_state.otp_notice = {
                    "kind": "error", "text": "Too many code requests. Please wait before trying again."
                }
                st.rerun()
            otp_value = generate_otp()
            ok, message = send_otp_email(st.session_state.reset_email, otp_value, purpose="reset")
            if ok:
                record_otp_request(st.session_state.reset_email)
                st.session_state.reset_token = make_otp_token(
                    st.session_state.reset_email, otp_value
                )
                st.session_state.otp_notice = {
                    "kind": "success", "text": "A new verification code has been sent."
                }
            else:
                st.session_state.otp_notice = {"kind": "error", "text": message}
            st.rerun()
        elif action == "change_email":
            clear_reset_flow()
            st.rerun()
        elif action == "signin":
            clear_reset_flow()
            go("Login")

    # Stage 3: Set a new password
    elif st.session_state.reset_stage == "password":
        st.markdown("""
        <style>
        .stApp, [data-testid="stAppViewContainer"], [data-testid="stHeader"] { background-color:#6352E0 !important; background-image:linear-gradient(to right,rgba(255,255,255,.08) 1px,transparent 1px),linear-gradient(to bottom,rgba(255,255,255,.08) 1px,transparent 1px),radial-gradient(circle at 50% 50%,#7260EE 0%,#523FD6 100%) !important; background-size:80px 80px,80px 80px,100% 100% !important; background-attachment:fixed !important; }
        header[data-testid="stHeader"] { display:none !important; }.main .block-container,[data-testid="stMainBlockContainer"] { max-width:100% !important;width:100% !important;padding:2rem 0 !important; } iframe[data-testid="stCustomComponentV1"] { width:100% !important;border:none !important; }
        </style>
        """, unsafe_allow_html=True)

        if not st.session_state.otp_verified:
            clear_reset_flow()
            st.rerun()

        event = reset_password_component(
            notice=st.session_state.get("reset_password_notice"),
            key="reset_password",
        )
        if event:
            event_id = event.get("event_id")
            if event_id and event_id == st.session_state.get("last_reset_password_event"):
                return
            st.session_state.last_reset_password_event = event_id
            action = event.get("action")
            if action == "signin":
                clear_reset_flow()
                go("Login")
            elif action == "update_password":
                new_password = str(event.get("new_password", ""))
                confirm = str(event.get("confirm_password", ""))
                if not new_password:
                    st.session_state.reset_password_notice = {
                        "kind": "error", "text": "Please enter a new password.", "field": "password"
                    }
                    st.rerun()
                _, _, _, _, checks = check_password_strength(new_password)
                if not checks["length"]:
                    st.session_state.reset_password_notice = {
                        "kind": "error", "text": "Password must be at least 8 characters long.", "field": "password"
                    }
                    st.rerun()
                if not (checks["uppercase"] and checks["lowercase"]):
                    st.session_state.reset_password_notice = {
                        "kind": "error", "text": "Password must contain both uppercase and lowercase letters.", "field": "password"
                    }
                    st.rerun()
                if not checks["digit"]:
                    st.session_state.reset_password_notice = {
                        "kind": "error", "text": "Password must contain at least one number (0-9).", "field": "password"
                    }
                    st.rerun()
                if not checks["special"]:
                    st.session_state.reset_password_notice = {
                        "kind": "error", "text": "Password must contain at least one special symbol (!@#$%^&*).", "field": "password"
                    }
                    st.rerun()
                if not confirm:
                    st.session_state.reset_password_notice = {
                        "kind": "error", "text": "Please confirm your new password.", "field": "confirm"
                    }
                    st.rerun()
                if new_password != confirm:
                    st.session_state.reset_password_notice = {
                        "kind": "error", "text": "Confirm Password does not match Password.", "field": "confirm"
                    }
                    st.rerun()
                update_password(st.session_state.reset_email, new_password)
                clear_reset_flow()
                go("Login")
        return

        page_header(
            "Set a new password",
            "Choose a strong, secure password for your account"
        )
        render_step_indicator("password")

        if not st.session_state.otp_verified:
            clear_reset_flow()
            st.error("Please verify a new code first.")
            st.rerun()

        st.markdown(
            textwrap.dedent("""
            <div style="
                background: rgba(16, 185, 129, 0.1);
                border: 1px solid rgba(16, 185, 129, 0.3);
                border-radius: 12px;
                padding: 14px 16px;
                margin-bottom: 16px;
                color: #A7F3D0;
                font-size: 0.9rem;
            ">
                ✅ <b>Email verified successfully!</b> Enter your new password below.
            </div>
            """),
            unsafe_allow_html=True
        )

        with st.form("new_password_form"):
            new_password = st.text_input(
                "New Password",
                type="password",
                placeholder="At least 8 characters with mix of case, numbers & symbols"
            )

            confirm = st.text_input(
                "Confirm New Password",
                type="password",
                placeholder="Re-enter password"
            )

            render_password_meter(new_password if "new_password" in locals() and new_password else "")

            submitted = st.form_submit_button(
                "Update Password →",
                use_container_width=True
            )

        if submitted:
            score, level, color, percent, checks = check_password_strength(new_password)

            if len(new_password) < 8:
                st.warning(
                    "Password must be at least 8 characters long."
                )

            elif not (checks["digit"] or checks["special"]):
                st.warning(
                    "Password should include at least one number or special character for security."
                )

            elif new_password != confirm:
                st.error("Passwords do not match.")

            else:
                update_password(
                    st.session_state.reset_email,
                    new_password
                )

                clear_reset_flow()

                st.success(
                    "Password updated successfully! Please sign in with your new password."
                )

                go("Login")

        st.markdown("<div style='height: 8px;'></div>", unsafe_allow_html=True)
        if st.button(
            "← Cancel and return to sign in",
            use_container_width=True,
            key="cancel_to_login"
        ):
            clear_reset_flow()
            go("Login")


def show_dashboard(user):
    """Renders User Dashboard (Step 10) or Admin Dashboard with Registered Users Directory (Step 11)."""
    import html
    import streamlit as st
    import pandas as pd

    if hasattr(user, "keys"):
        user = {key: user[key] for key in user.keys()}

    username = html.escape(str(user.get("username", "User")))
    email = html.escape(str(user.get("email", "")))
    role = str(user.get("role", "user")).lower()
    is_admin = (role == "admin")

    st.markdown(
        """
        <style>
        [data-testid="stAppViewContainer"] {
            background: linear-gradient(135deg, #0b1020 0%, #111827 55%, #21113b 100%) !important;
        }
        [data-testid="stHeader"] { background: transparent !important; }
        .welcome-card {
            box-sizing: border-box;
            width: min(100%, 1060px);
            margin: 3vh auto 20px auto;
            padding: 42px 28px;
            text-align: center;
            background: #ffffff;
            border: 1px solid rgba(255,255,255,.8);
            border-radius: 24px;
            box-shadow: 0 24px 70px rgba(0, 0, 0, .28);
        }
        .admin-badge {
            display: inline-block;
            background: #fdf2f8;
            color: #be185d;
            border: 1px solid #fbcfe8;
            font-size: 0.8rem;
            font-weight: 700;
            text-transform: uppercase;
            letter-spacing: 0.08em;
            padding: 4px 12px;
            border-radius: 9999px;
            margin-bottom: 14px;
        }
        .user-badge {
            display: inline-block;
            background: #ecfdf5;
            color: #047857;
            border: 1px solid #a7f3d0;
            font-size: 0.8rem;
            font-weight: 700;
            text-transform: uppercase;
            letter-spacing: 0.08em;
            padding: 4px 12px;
            border-radius: 9999px;
            margin-bottom: 14px;
        }
        .welcome-icon {
            width: 70px;
            height: 70px;
            margin: 0 auto 20px auto;
            border-radius: 50%;
            display: flex;
            align-items: center;
            justify-content: center;
            color: #ffffff !important;
            font-size: 36px;
            line-height: 1;
            background: linear-gradient(135deg, #6366f1, #9333ea);
        }
        .welcome-card .welcome-title {
            display: block !important;
            margin: 0 0 10px 0 !important;
            color: #172033 !important;
            font-size: clamp(26px, 3vw, 34px) !important;
            font-weight: 800 !important;
            line-height: 1.25 !important;
        }
        .welcome-card .welcome-copy {
            margin: 0 !important;
            color: #52627a !important;
            font-size: 16px !important;
            line-height: 1.6 !important;
        }
        .admin-directory-card {
            width: min(100%, 1060px);
            margin: 0 auto 30px auto;
            background: #ffffff;
            border-radius: 20px;
            padding: 30px;
            box-shadow: 0 20px 60px rgba(0, 0, 0, .25);
            color: #172033;
        }
        div[data-testid="stButton"] button {
            border-radius: 12px !important;
            min-height: 44px !important;
            font-weight: 650 !important;
        }
        </style>
        """,
        unsafe_allow_html=True,
    )

    if is_admin:
        # STEP 11: Admin Dashboard with Registered Users List
        st.markdown(
            f"""
            <div class="welcome-card">
                <div class="admin-badge">🛡️ Administrator Console</div>
                <div class="welcome-icon">⚡</div>
                <div class="welcome-title">Welcome back, {username}!</div>
                <p class="welcome-copy">
                    Logged in as Administrator: <strong>{email}</strong><br>
                    You have elevated access to view registered accounts in the user directory below.
                </p>
            </div>
            """,
            unsafe_allow_html=True,
        )

        users_list = get_all_users_for_admin()
        user_rows = [
            {
                "Username": row["username"],
                "Email": row["email"],
                "Role": row["role"].upper(),
                "Registered At": row["created_at"]
            }
            for row in users_list
        ]

        df_users = pd.DataFrame(user_rows)

        col_left, col_mid, col_right = st.columns([1, 10, 1])
        with col_mid:
            st.markdown(
                """
                <div style="background: rgba(255,255,255,0.06); border: 1px solid rgba(255,255,255,0.12); border-radius: 16px; padding: 22px 24px; margin-bottom: 24px;">
                    <div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:12px; margin-bottom:16px;">
                        <div>
                            <h3 style="margin:0; font-size:1.3rem; color:#FFFFFF;">👥 Registered Users Directory</h3>
                            <p style="margin:4px 0 0; color:#94A3B8; font-size:0.88rem;">Complete list of registered users. Password hashes are strictly omitted.</p>
                        </div>
                    </div>
                </div>
                """,
                unsafe_allow_html=True
            )

            col_stat1, col_stat2 = st.columns(2)
            with col_stat1:
                st.metric("Total Registered Users", len(user_rows))
            with col_stat2:
                admin_count = sum(1 for u in user_rows if u["Role"] == "ADMIN")
                st.metric("Administrators", admin_count)

            st.markdown("<div style='height: 10px;'></div>", unsafe_allow_html=True)
            search_query = st.text_input("🔍 Search users by username or email", placeholder="Filter users...")

            if search_query.strip():
                q = search_query.strip().lower()
                filtered_rows = [
                    u for u in user_rows
                    if q in u["Username"].lower() or q in u["Email"].lower()
                ]
                df_to_show = pd.DataFrame(filtered_rows)
            else:
                df_to_show = df_users

            st.dataframe(df_to_show, use_container_width=True, hide_index=True)

    else:
        # STEP 10: Standard User Dashboard
        st.markdown(
            f"""
            <div class="welcome-card">
                <div class="user-badge">✓ Active Member</div>
                <div class="welcome-icon">✓</div>
                <div class="welcome-title">Welcome back, {username}!</div>
                <p class="welcome-copy">
                    Logged-in Email: <strong>{email}</strong><br>
                    You are securely authenticated to the Security Gateway. Have a productive day!
                </p>
            </div>
            """,
            unsafe_allow_html=True,
        )

    # Logout Action (Step 10 & Step 11)
    st.markdown("<div style='height: 16px;'></div>", unsafe_allow_html=True)
    _, center_col, _ = st.columns([1, 1, 1])
    with center_col:
        if st.button("🚪 Sign out", use_container_width=True, key="dashboard_signout_btn"):
            st.session_state.login_token = None
            st.session_state.page = "Login"
            clear_reset_flow()
            st.rerun()


In [ ]:
# @title Application
%%writefile app.py
import streamlit as st

st.set_page_config(page_title="Security Gateway", layout="wide")
from config import JWT_SECRET, apply_theme
from database import init_db, get_user
from auth import decode_login_token
from ui import (
    show_login,
    show_signup,
    show_forgot,
    show_dashboard,
    clear_reset_flow
)

apply_theme()
init_db()

# Initialize application session state.
DEFAULTS = {
    "page": "Login",
    "login_token": None,
    "reset_email": "",
    "reset_token": None,
    "reset_stage": "email",
    "otp_verified": False,
}

for key, value in DEFAULTS.items():
    if key not in st.session_state:
        st.session_state[key] = value

if not JWT_SECRET:
    st.warning(
        "Add a strong JWT_SECRET to Google Colab Secrets and restart."
    )

# Validate login session.
payload = decode_login_token(st.session_state.login_token)

if st.session_state.login_token and not payload:
    st.session_state.login_token = None
    st.session_state.page = "Login"
    st.rerun()

# Route authenticated users

if payload:
    current_user = get_user(payload.get("sub", ""))
    if not current_user:
        st.session_state.login_token = None
        st.session_state.page = "Login"
        st.rerun()
    show_dashboard(current_user)

# Route unauthenticated users
else:
    col_ratio = [1, 2.5, 1] if st.session_state.page in ["Forgot", "Signup"] else [1, 1.6, 1]
    _, center, _ = st.columns(col_ratio)

    with center:
        page = st.session_state.page

        if page == "Login":
            show_login()
        elif page == "Signup":
            show_signup()
        elif page == "Forgot":
            show_forgot()
        else:
            st.session_state.page = "Login"
            st.rerun()


In [ ]:
# @title Lauch App
# Final Step: Launch Streamlit with public ngrok tunnel
import os
import time
import subprocess
from pyngrok import ngrok
from google.colab import userdata

# 1. Forward all Colab Secrets into environment variables for Streamlit
for secret_name in ["JWT_SECRET", "NGROK_AUTHTOKEN", "EMAIL_ADDRESS", "EMAIL_ID", "EMAIL_PASSWORD", "ADMIN_EMAIL", "ADMIN_PASSWORD"]:
    try:
        val = userdata.get(secret_name)
        if val is not None:
            os.environ[secret_name] = str(val)
    except Exception:
        pass

NGROK_TOKEN = os.environ.get("NGROK_AUTHTOKEN")
if not NGROK_TOKEN:
    raise ValueError("Please configure NGROK_AUTHTOKEN in Colab Secrets and enable the Notebook access toggle.")

ngrok.set_auth_token(NGROK_TOKEN)

# 2. Kill any existing ngrok tunnels or streamlit sessions
ngrok.kill()
!pkill -f streamlit

# 3. Start Streamlit in the background on port 8501
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(3)

# 4. Open ngrok tunnel
public_url = ngrok.connect(8501).public_url
print("=" * 60)
print(f"Infosys Portal Live URL: {public_url}")
print("=" * 60)
print("App is running! Press the Stop button in this cell to shut down.")

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\nShutting down...")
    ngrok.kill()
    process.terminate()
    !pkill -f streamlit
    print("Ngrok tunnel closed and Streamlit server stopped gracefully.")
